In [ ]:
# === Colab rescue scaffolding: mount Drive + tee stdout/stderr + checkpoints ====
# Streams every log line to Google Drive so an interrupted T4 session keeps its
# evidence. No token is hard-coded. Safe to skip if Drive is unavailable.
import os, sys, json, time, atexit, shutil
from pathlib import Path

CALM_RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
CALM_DRIVE_DIR = None
CALM_LOCAL_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

class _Tee:
    def __init__(self, stream, path):
        self.stream = stream; self.file_path = Path(path)
        self.file_path.parent.mkdir(parents=True, exist_ok=True)
        self.file = open(self.file_path, "a", encoding="utf-8", buffering=1)
        self._calm_tee = True; self.encoding = getattr(stream, "encoding", "utf-8")
    def write(self, data):
        try: self.stream.write(data)
        except Exception: pass
        try: self.file.write(data); self.file.flush()
        except Exception: pass
    def flush(self):
        for t in (self.stream, self.file):
            try: t.flush()
            except Exception: pass
    def isatty(self): return False
    def __getattr__(self, name): return getattr(self.stream, name)

def _mount_drive():
    global CALM_DRIVE_DIR
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        root = Path("/content/drive/MyDrive/calm_fuzz_12b_runs")
        CALM_DRIVE_DIR = root / CALM_RUN_STAMP
        CALM_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        return CALM_DRIVE_DIR
    except Exception as e:
        CALM_DRIVE_DIR = None
        print(">> No Google Drive; keeping artifacts in the runtime only:", str(e)[:140])
        return None

def calm_local(name): return CALM_LOCAL_DIR / name
def calm_drive(name): return (Path(CALM_DRIVE_DIR) / name) if CALM_DRIVE_DIR else None
def calm_copy_to_drive(path, name=None):
    if not CALM_DRIVE_DIR: return None
    src = Path(path)
    if not src.exists(): return None
    dst = Path(CALM_DRIVE_DIR) / (name or src.name)
    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); return str(dst)

_dd = _mount_drive()
if _dd:
    if not getattr(sys.stdout, "_calm_tee", False): sys.stdout = _Tee(sys.stdout, _dd / "stdout_stream.log")
    if not getattr(sys.stderr, "_calm_tee", False): sys.stderr = _Tee(sys.stderr, _dd / "stderr_stream.log")
    atexit.register(lambda: getattr(sys.stdout, "flush", lambda: None)())
    print(">> Rescue stream -> Drive:", _dd)
else:
    print(">> No Drive stream. Re-run this cell and allow mount if you want crash-proof logs.")


Mounted at /content/drive
>> Rescue stream -> Drive: /content/drive/MyDrive/calm_fuzz_12b_runs/20260610_233749


In [ ]:
# === Environment: deps, MOCK/torch/CUDA detection, Hugging Face login ==========
import os, sys, subprocess, importlib.util, re
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

def _has(m): return importlib.util.find_spec(m) is not None
def _ver(pkg):
    try:
        from importlib.metadata import version; return version(pkg)
    except Exception: return "0"
def _vparts(v):
    p = [int(x) for x in re.findall(r"\d+", str(v))[:3]]; return tuple(p + [0] * (3 - len(p)))
def _pip(specs):
    if specs:
        print(">> install:", " ".join(specs))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *specs], check=False)

# core
_pip([p for p in ["numpy", "pandas", "matplotlib"] if not _has(p)])

# Toggle: set FUZZ_MOCK=1 to smoke-test the whole pipeline on CPU with no model.
MOCK = os.environ.get("FUZZ_MOCK", "0") == "1"
HAVE_TORCH = _has("torch")
HAS_CUDA = False
if HAVE_TORCH and not MOCK:
    try:
        import torch; HAS_CUDA = torch.cuda.is_available()
    except Exception: HAS_CUDA = False
USE_LLM = (not MOCK) and HAS_CUDA   # Gemma only when a real GPU is present

if USE_LLM:
    specs = []
    if _vparts(_ver("transformers")) < _vparts("5.10.1"): specs.append("transformers>=5.10.1")
    for pkg, mod in [("accelerate", "accelerate"), ("bitsandbytes", "bitsandbytes"),
                     ("huggingface_hub", "huggingface_hub"), ("sentencepiece", "sentencepiece"),
                     ("protobuf", "google.protobuf"), ("safetensors", "safetensors")]:
        if not _has(mod): specs.append(pkg)
    _pip(specs)

import numpy as np
print(">> MOCK:", MOCK, "| torch:", HAVE_TORCH, "| cuda:", HAS_CUDA, "| use_llm:", USE_LLM)

# Hugging Face token from Colab secret or env (never hard-code it)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
        if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
    except Exception: pass
if HF_TOKEN and USE_LLM:
    try:
        from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False)
        print(">> Logged in to Hugging Face via HF_TOKEN.")
    except Exception as e:
        print(">> HF login error:", str(e)[:120])
elif USE_LLM:
    print(">> No HF_TOKEN yet; add a Colab secret named HF_TOKEN if you hit 401/403.")


>> install: bitsandbytes
>> MOCK: False | torch: True | cuda: True | use_llm: True
>> No HF_TOKEN yet; add a Colab secret named HF_TOKEN if you hit 401/403.


In [ ]:
# -*- coding: utf-8 -*-
"""
CALM-Fuzz engine  (Calibrated oracle + Amortized-LLM policy + MAP-Elites coverage)
==================================================================================
Reliability / differential-testing harness for deep-learning tensor operators.

Standalone proof-on-CPU. A pure-numpy "mock framework" has CONTROLLED fault
injection (5 fault classes) plus realistic per-device float noise, so we can
measure oracle precision/recall against ground truth -- and contrast it with the
old fingerprint oracle -- without any GPU. The notebook swaps the mock executor
for a real torch CPU/CUDA executor and the mock policy for Gemma 4 12B; the
orchestration is identical.

Controlled fault injection (each keyed to seed-tensor + HEAD op, so ground truth
is unambiguous and every reported reproducer minimizes to 2 steps):
  - crash               : conv2d(k<=0) on a valid 4D tensor
  - contract_drift      : inv() of a singular (all-zeros) square matrix
                          -> CPU raises, device path returns inf  (exception contract)
  - device_divergence   : cumsum on float32 long axis -> device result drifts 5e-3
  - compile_drift       : exp on 'big' values -> compiled path overflows, eager clamps
  - metamorphic_violation: softmax on a wide axis breaks shift-invariance by 5e-3

Everything else is benign: per-device float noise ~1e-6, expected overflow from
'big'/'inf'/'nan' fills, and normal shape errors. A good oracle must stay silent
on all of those.
"""
import os, sys, json, time, math, random, hashlib
from collections import Counter, defaultdict

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
import numpy as np
np.seterr(all="ignore")  # overflow/invalid are EXPECTED here; the oracle decides, not warnings


# --------------------------------------------------------------------------- #
# 0. Config
# --------------------------------------------------------------------------- #
class CFG:
    time_budget_s     = 6.0
    rng_seed          = 7
    numel_cap         = 200_000
    calib_samples     = 60
    tol_safety        = 8.0
    tol_floor         = 1e-4
    llm_burst_every_s = 1.5
    corpus_cap        = 400
    print_every       = 0.7
    checkpoint_every  = 0          # >0 -> call checkpoint_cb every N tests (notebook sets this)

DTYPES = ["float32", "float64", "int32", "int64"]
FILLS  = ["normal", "zeros", "nan", "inf", "big"]
OPS    = ["matmul", "softmax", "reshape", "conv2d", "inv", "cumsum", "relu", "add", "exp"]
MR_OPS = {"softmax", "matmul", "relu", "exp", "reshape", "cumsum"}
COMPILE_SENSITIVE = {"exp"}   # only run the eager/compiled oracle when these appear


class CrashSignal(Exception):
    """Stands in for a process-killing fault (SIGSEGV/abort) the real harness
    would observe via a subprocess exit code."""


# --------------------------------------------------------------------------- #
# 1. Program DSL  (same shape as the baseline notebook)
# --------------------------------------------------------------------------- #
def _phash(prog):
    return int(hashlib.blake2b(json.dumps(prog, sort_keys=True).encode(), digest_size=8).hexdigest(), 16)

def numel(shape):
    n = 1
    for d in shape:
        n *= int(d)
    return n

def _seed(prog):
    s = prog.get("steps", [])
    return s[0] if s and s[0].get("op") == "tensor" else {}

def _ops(prog):
    return [s.get("op") for s in prog.get("steps", []) if s.get("op") != "tensor"]

def _head(prog):
    s = prog.get("steps", [])
    return s[1] if len(s) > 1 and s[0].get("op") == "tensor" else {}


# --------------------------------------------------------------------------- #
# 2. Probabilistic grammar / policy  (the fast, GPU-free sampler)
# --------------------------------------------------------------------------- #
class Policy:
    def __init__(self):
        self.op_w    = {o: 1.0 for o in OPS}
        self.fill_w  = {"normal": 7, "zeros": 2, "nan": 1, "inf": 1, "big": 1}
        self.dtype_w = {"float32": 5, "float64": 2, "int32": 1, "int64": 1}
        self.rank_w  = {1: 1, 2: 5, 3: 2, 4: 1}
        self.edge_p  = 0.16
        self.big_dim = 10**8
        self.source_mix = {"seed": 0.10, "mutate": 0.35, "grammar": 0.55}
        self.seed_queue = []
        self.patches = 0

    @staticmethod
    def _pick(rng, wd):
        ks = list(wd.keys()); ws = [max(1e-6, wd[k]) for k in ks]
        return rng.choices(ks, weights=ws, k=1)[0]

    def _shape(self, rng, target=None):
        if target == "conv2d":
            shp = [rng.randint(1, 2), rng.randint(1, 3), rng.randint(2, 12), rng.randint(2, 12)]
        elif target in ("matmul", "inv"):
            n = rng.randint(1, 8); shp = [n, n]
        elif target in ("softmax", "cumsum", "relu", "add", "exp"):
            shp = [rng.randint(1, 6), rng.randint(1, 40)]
        else:
            shp = [rng.randint(1, 8) for _ in range(self._pick(rng, self.rank_w))]
        if shp and rng.random() < self.edge_p:
            i = rng.randrange(len(shp)); shp[i] = rng.choice([0, -rng.randint(1, 3), self.big_dim])
        return shp

    def seed_step(self, rng, target=None):
        return {"op": "tensor", "shape": self._shape(rng, target),
                "dtype": self._pick(rng, self.dtype_w), "fill": self._pick(rng, self.fill_w)}

    def sample(self, rng, target=None):
        steps = [self.seed_step(rng, target)]
        for i in range(rng.choice([1, 1, 2, 3])):
            o = target if (i == 0 and target in self.op_w and rng.random() < 0.8) else self._pick(rng, self.op_w)
            st = {"op": o}
            if o == "reshape": st["target"] = rng.choice([[-1], [1, -1], [rng.randint(1, 8), -1]])
            if o == "conv2d":  st["k"] = rng.choice([-1, 0, 1, 3])
            steps.append(st)
        return {"steps": steps}

    def mutate(self, prog, rng, target=None):
        p = json.loads(json.dumps(prog)); steps = p.get("steps") or [self.seed_step(rng, target)]
        s = rng.choice(steps)
        if s.get("op") == "tensor":
            k = rng.choice(["shape", "dtype", "fill"])
            s[k] = (self._shape(rng, target) if k == "shape"
                    else self._pick(rng, self.dtype_w if k == "dtype" else self.fill_w))
        elif rng.random() < 0.5:
            s["op"] = self._pick(rng, self.op_w)
        else:
            steps.append({"op": self._pick(rng, self.op_w)})
        p["steps"] = steps
        return p

    def build_from_seq(self, seq, rng):
        target = next((o for o in seq if o in OPS), None)
        steps = [self.seed_step(rng, target)]
        for o in seq:
            if o in OPS:
                st = {"op": o}
                if o == "reshape": st["target"] = [-1]
                if o == "conv2d":  st["k"] = rng.choice([0, 1, 3])
                steps.append(st)
        return {"steps": steps}

    def apply_patch(self, patch):
        """Bounded, whitelisted update so a bad LLM reply cannot break the run."""
        self.patches += 1
        for o in patch.get("boost_ops", []):
            if o in self.op_w: self.op_w[o] = min(8.0, self.op_w[o] * 1.6 + 0.5)
        for o in patch.get("cool_ops", []):
            if o in self.op_w: self.op_w[o] = max(0.15, self.op_w[o] * 0.6)
        for f in patch.get("boost_fills", []):
            if f in self.fill_w: self.fill_w[f] = min(9.0, self.fill_w[f] * 1.8 + 0.6)
        if "edge_p" in patch:
            try: self.edge_p = float(np.clip(float(patch["edge_p"]), 0.0, 0.6))
            except Exception: pass
        for seq in patch.get("try_sequences", [])[:8]:
            if isinstance(seq, list) and all(isinstance(x, str) for x in seq):
                self.seed_queue.append([x for x in seq if x in OPS or x == "tensor"])


In [ ]:
# --------------------------------------------------------------------------- #
# 3. Mock framework: numpy compute + controlled fault injection + device noise
# --------------------------------------------------------------------------- #
def gt_kind(prog):
    """Ground-truth fault class, keyed to (seed tensor, HEAD op). '' = benign.
    Requires a constructible, all-positive, non-huge seed so the head op is
    actually reached."""
    sd = _seed(prog); hd = _head(prog); op = hd.get("op")
    shp = sd.get("shape", []); fill = sd.get("fill", "normal"); dt = sd.get("dtype", "float32")
    if not shp or any((not isinstance(d, int)) or d <= 0 for d in shp): return ""
    if numel(shp) > CFG.numel_cap: return ""
    if op == "conv2d" and int(hd.get("k", 1)) <= 0 and len(shp) == 4:                  return "crash"
    if op == "inv" and fill == "zeros" and len(shp) == 2 and shp[0] == shp[1]:         return "contract_drift"
    if op == "cumsum" and dt == "float32" and isinstance(shp[-1], int) and shp[-1] >= 16: return "device_divergence"
    if op == "exp" and fill == "big":                                                  return "compile_drift"
    if op == "softmax" and isinstance(shp[-1], int) and shp[-1] >= 32 and fill == "normal": return "metamorphic_violation"
    return ""


def _mk(shape, dtype, fill):
    if any((not isinstance(d, int)) or d < 0 for d in shape):
        raise ValueError("bad dim")
    np_dt = {"float32": np.float32, "float64": np.float64, "int32": np.int32, "int64": np.int64}[dtype]
    fdt = np_dt if "float" in dtype else np.float32
    n = numel(shape) if shape else 1
    if fill == "nan":   return np.full(shape, np.nan, dtype=fdt)
    if fill == "inf":   return np.full(shape, np.inf, dtype=fdt)
    if fill == "big":   return np.full(shape, 1e30 if "float" in dtype else 2**30, dtype=np_dt)
    if fill == "zeros": return np.zeros(shape, dtype=np_dt)
    return ((np.arange(n, dtype=np.float64).reshape(shape) % 17 - 8) / 8.0).astype(np_dt)


def _apply(v, st, device, path, trig):
    op = st.get("op")
    if op == "matmul":   return np.matmul(v, v)
    if op == "softmax":
        x = v.astype(np.float64); x = x - x.max(axis=-1, keepdims=True)
        e = np.exp(x); return (e / e.sum(axis=-1, keepdims=True)).astype(v.dtype)
    if op == "reshape":  return v.reshape(st.get("target", [-1]))
    if op == "conv2d":
        k = int(st.get("k", 1))
        if k <= 0:
            if trig == "crash": raise CrashSignal("conv2d kernel<=0")
            raise ValueError("conv2d kernel<=0")
        if v.ndim != 4: raise ValueError("conv2d needs 4D")
        return v[:, :, :-1 if v.shape[2] > 1 else None, :]
    if op == "inv":
        if trig == "contract_drift" and device == "cuda":
            try: return np.linalg.inv(v)
            except np.linalg.LinAlgError: return np.full_like(v, np.inf)
        return np.linalg.inv(v)                       # both devices raise on singular -> invalid
    if op == "cumsum":
        out = np.cumsum(v, axis=-1)
        if trig == "device_divergence" and device == "cuda": out = out * (1.0 + 5e-3)
        return out
    if op == "relu":     return np.maximum(v, 0)
    if op == "add":      return v + v
    if op == "exp":
        x = v.astype(np.float64)
        if trig == "compile_drift" and path == "compiled":
            return np.exp(x).astype(v.dtype)          # compiled overflows -> inf
        return np.exp(np.clip(x, -50, 50)).astype(v.dtype)  # eager clamps -> stays finite even in fp32
    raise ValueError("unknown op " + str(op))


def run_program(prog, device="cpu", path="eager"):
    trig = gt_kind(prog); v = None
    for st in prog.get("steps", []):
        if st.get("op") == "tensor":
            try: v = _mk(st.get("shape", []), st.get("dtype", "float32"), st.get("fill", "normal"))
            except Exception as e: return {"status": "invalid", "etype": type(e).__name__}
            continue
        if v is None: return {"status": "invalid", "etype": "no_tensor"}
        try: v = _apply(v, st, device, path, trig)
        except CrashSignal as e: return {"status": "crash", "etype": "CrashSignal", "msg": str(e)}
        except Exception as e:   return {"status": "exception", "etype": type(e).__name__}
    if v is None: return {"status": "invalid", "etype": "empty"}
    # benign per-device noise that NEVER flips finiteness (so it can't fake a bug)
    if device == "cuda" and np.issubdtype(np.asarray(v).dtype, np.floating):
        rng = np.random.default_rng(_phash(prog) & 0xFFFFFFFF)
        noisy = v * (1.0 + rng.normal(0.0, 1e-6, size=np.shape(v)))
        flip = np.isfinite(v) & ~np.isfinite(noisy)
        v = np.where(flip, v, noisy).astype(np.asarray(v).dtype)
    return {"status": "ok", "val": np.asarray(v)}


# ---- metamorphic relations -------------------------------------------------- #
def _mr_seed(prog):
    sd = _seed(prog)
    try: v = _mk(sd.get("shape", []), sd.get("dtype", "float32"), sd.get("fill", "normal"))
    except Exception: return None
    return v.astype(np.float64) if not np.issubdtype(v.dtype, np.floating) else v

def mr_softmax(v, trig):
    if v.ndim < 1 or v.shape[-1] < 2: return None
    def sm(x):
        x = x - x.max(axis=-1, keepdims=True); e = np.exp(x); return e / e.sum(axis=-1, keepdims=True)
    lhs, rhs = sm(v), sm(v - 3.0)
    if trig == "metamorphic_violation": rhs = rhs * (1.0 + 5e-3)
    return lhs, rhs
def mr_relu(v, trig):   return np.maximum(v, 0) + np.maximum(-v, 0), np.abs(v)
def mr_exp(v, trig):    x = np.clip(v, -50, 50); return np.exp(x + 0.5), np.exp(x) * math.exp(0.5)
def mr_matmul(v, trig):
    if v.ndim != 2 or v.shape[0] != v.shape[1] or v.shape[0] == 0: return None
    return (v @ v) @ v, v @ (v @ v)
def mr_reshape(v, trig): return v.reshape(-1).reshape(v.shape), v
def mr_cumsum(v, trig):
    if v.ndim < 1 or v.shape[-1] < 2: return None
    return np.diff(np.cumsum(v, axis=-1), axis=-1, prepend=0.0), v
MR_TABLE = {"softmax": mr_softmax, "relu": mr_relu, "exp": mr_exp,
            "matmul": mr_matmul, "reshape": mr_reshape, "cumsum": mr_cumsum}


# --------------------------------------------------------------------------- #
# 3b. Executor interface
#   .run(prog, device, path) -> {status, val}
#   .metamorphic(prog)       -> [(op, rel_diff), ...]
# The notebook supplies a TorchExecutor with the same interface (subprocess
# worker for crash isolation); orchestration + oracle verdict are shared.
# --------------------------------------------------------------------------- #
class MockExecutor:
    name = "mock"; has_ground_truth = True
    def run(self, prog, device="cpu", path="eager"):
        return run_program(prog, device, path)
    def metamorphic(self, prog):
        out = []
        for op in set(_ops(prog)) & MR_OPS:
            v = _mr_seed(prog)
            if v is None: continue
            pair = MR_TABLE[op](v, gt_kind(prog))
            if pair is None: continue
            lhs, rhs = pair
            if not (np.isfinite(lhs).all() and np.isfinite(rhs).all()): continue
            out.append((op, rel_diff(lhs, rhs)))
        return out


In [ ]:
# --------------------------------------------------------------------------- #
# 4. Calibrated oracle (measure = framework-specific ; verdict = shared rules)
# --------------------------------------------------------------------------- #
def rel_diff(a, b):
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    if a.shape != b.shape: return math.inf
    fa, fb = np.isfinite(a), np.isfinite(b)
    if not np.array_equal(fa, fb): return math.inf
    m = fa & fb
    if not m.any(): return 0.0
    return float(np.max(np.abs(a[m] - b[m]) / (np.abs(b[m]) + 1e-12)))


class Oracle:
    AXIOM_BUGS = {"axiom_bit_symmetry", "axiom_scale_symmetry", "axiom_canary_lane", "axiom_envelope"}
    BUGS = {"device_divergence", "metamorphic_violation", "contract_drift", "compile_drift",
            "crash", "hang"} | AXIOM_BUGS
    NUMERIC_GRAY_OPS = {"inv", "exp", "softmax"}

    def __init__(self, cfg, executor):
        self.cfg = cfg; self.ex = executor; self.tol = {}; self.global_tol = cfg.tol_floor

    def _is_truth_mode(self):
        return bool(getattr(self.ex, "has_ground_truth", False))

    def _strict_contract(self):
        return self._is_truth_mode() or bool(getattr(self.cfg, "enable_real_contract_oracle", False))

    def _strict_compile(self):
        return self._is_truth_mode() or bool(getattr(self.cfg, "enable_real_compile_oracle", False))

    def _strict_numeric_drift(self, prog):
        if self._is_truth_mode() or bool(getattr(self.cfg, "enable_real_heuristic_drift_oracle", False)):
            return True
        return not (set(_ops(prog)) & self.NUMERIC_GRAY_OPS)

    # ---- framework-specific MEASUREMENT (raw, tol-free) ----------------- #
    def measure(self, prog):
        if hasattr(self.ex, "measure_fused"):     # torch worker: one IPC round-trip
            return self.ex.measure_fused(prog)
        rc = self.ex.run(prog, "cpu"); rg = self.ex.run(prog, "cuda")
        m = {"cpu": rc["status"], "cuda": rg["status"], "etype": rc.get("etype", "") or rg.get("etype", ""),
             "cpu_etype": rc.get("etype", ""), "cuda_etype": rg.get("etype", ""),
             "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}
        if rc["status"] == "ok" and rg["status"] == "ok":
            m["dev_rel"] = rel_diff(rc["val"], rg["val"])
        if COMPILE_SENSITIVE & set(_ops(prog)):
            re, rk = self.ex.run(prog, "cpu", "eager"), self.ex.run(prog, "cpu", "compiled")
            if re["status"] == "ok" and rk["status"] == "ok":
                m["compile_rel"] = rel_diff(re["val"], rk["val"])
        m["mr"] = self.ex.metamorphic(prog)
        return m

    # ---- SHARED verdict rules (mock and torch use the identical logic) -- #
    def verdict(self, prog, m, tol=None):
        tol = self._tol(prog) if tol is None else tol
        sc, sg = m.get("cpu"), m.get("cuda")
        if sc in ("crash", "hang") or sg in ("crash", "hang"):
            lab = "hang" if (sc == "hang" or sg == "hang") else "crash"
            return {"status": lab, "detail": ("cpu" if sc in ("crash", "hang") else "cuda")}
        if (sc == "exception") != (sg == "exception"):
            detail = "cpu_raises" if sc == "exception" else "cuda_raises"
            et = "%s/%s" % (m.get("cpu_etype", ""), m.get("cuda_etype", ""))
            if self._strict_contract():
                return {"status": "contract_drift", "detail": "%s %s" % (detail, et)}
            return {"status": "invalid", "detail": "contract_mismatch_suppressed:%s %s" % (detail, et)}
        if sc in ("invalid", "exception") and sg in ("invalid", "exception"):
            return {"status": "invalid", "detail": m.get("etype", "")}

        # Theorem-style AXIOMplus probes are allowed to speak even when the
        # heuristic real-torch drift oracles are conservative.
        for ax in m.get("axioms", []) or []:
            if isinstance(ax, dict) and ax.get("ok") is False:
                return {"status": ax.get("status", "axiom_violation"), "detail": ax.get("detail", "")}

        if m.get("compile_rel") is not None and m["compile_rel"] > tol:
            if self._strict_compile():
                return {"status": "compile_drift", "detail": "eager!=alt rel=%.1e" % m["compile_rel"]}
            return {"status": "gray_numeric_drift", "detail": "compile_proxy_suppressed rel=%.1e" % m["compile_rel"]}
        for op, rel in m.get("mr", []) or []:
            if rel > max(tol, 1e-4):
                return {"status": "metamorphic_violation", "detail": "MR:%s rel=%.1e" % (op, rel)}
        if m.get("dev_rel") is not None and m["dev_rel"] > tol:
            if self._strict_numeric_drift(prog):
                return {"status": "device_divergence", "detail": "rel=%.1e>tol=%.0e" % (m["dev_rel"], tol)}
            return {"status": "gray_numeric_drift", "detail": "heuristic_suppressed rel=%.1e>tol=%.0e" % (m["dev_rel"], tol)}
        return {"status": "ok"}

    def evaluate(self, prog):
        return self.verdict(prog, self.measure(prog))

    def calibrate(self, policy, rng):
        gt_filter = getattr(self.ex, "has_ground_truth", False)
        obs = defaultdict(list); allobs = []; n = tries = 0
        while n < self.cfg.calib_samples and tries < self.cfg.calib_samples * 15:
            tries += 1
            prog = policy.sample(rng, rng.choice(OPS))
            if gt_filter and gt_kind(prog): continue
            sd = _seed(prog)
            if numel(sd.get("shape", []) or [1]) > self.cfg.numel_cap: continue
            m = self.measure(prog)
            if m.get("cpu") != "ok" or m.get("cuda") != "ok" or m.get("dev_rel") is None or not math.isfinite(m.get("dev_rel")):
                continue
            key = ((_ops(prog) or ["tensor"])[-1], sd.get("dtype", "float32"))
            obs[key].append(m["dev_rel"]); allobs.append(m["dev_rel"]); n += 1
        for k, v in obs.items():
            self.tol[k] = max(self.cfg.tol_floor, max(v) * self.cfg.tol_safety)
        if allobs: self.global_tol = max(self.cfg.tol_floor, max(allobs) * self.cfg.tol_safety)
        return n, self.global_tol

    def _tol(self, prog):
        sd = _seed(prog)
        return self.tol.get(((_ops(prog) or ["tensor"])[-1], sd.get("dtype", "float32")), self.global_tol)

    # OLD baseline oracle (abs-sum fingerprint + any-nonfinite) for contrast
    def baseline_verdict(self, prog):
        rc, rg = self.ex.run(prog, "cpu"), self.ex.run(prog, "cuda")
        if rc["status"] == "crash" or rg["status"] == "crash": return "crash"
        if rc["status"] in ("exception", "invalid") and rg["status"] in ("exception", "invalid"): return "invalid"
        vc, vg = rc.get("val"), rg.get("val")
        for vv in (vc, vg):
            if vv is not None and np.issubdtype(np.asarray(vv).dtype, np.floating) \
               and not np.isfinite(np.asarray(vv, np.float64)).all():
                return "silent_nonfinite"
        def fp(v):
            if v is None: return "na"
            return round(float(np.abs(np.asarray(v, np.float64)).sum()) % 1e6, 2)
        return "device_divergence" if fp(vc) != fp(vg) else "ok"


# --------------------------------------------------------------------------- #
# 5. MAP-Elites coverage archive
# --------------------------------------------------------------------------- #
def descriptor(prog):
    sd = _seed(prog); shp = sd.get("shape", [])
    bad = any(isinstance(d, int) and (d <= 0 or d >= 10**7) for d in shp)
    size_cls = "huge" if numel(shp or [1]) > CFG.numel_cap else ("edge" if bad else "small")
    return ((_ops(prog) or ["tensor"])[-1], sd.get("dtype", "float32"),
            sd.get("fill", "normal"), min(len(shp), 4), size_cls)

class Archive:
    RANK = {"crash": 6, "hang": 6, "contract_drift": 5,
            "axiom_bit_symmetry": 5, "axiom_scale_symmetry": 5,
            "axiom_canary_lane": 5, "axiom_envelope": 5,
            "device_divergence": 4, "metamorphic_violation": 4, "compile_drift": 4,
            "gray_numeric_drift": 2, "ok": 1}
    def __init__(self): self.cells = {}
    def consider(self, prog, status):
        cell = descriptor(prog); novel = cell not in self.cells
        r = self.RANK.get(status, 0); cur = self.cells.get(cell)
        if cur is None or r > cur[0]: self.cells[cell] = (r, prog)
        return novel
    def gaps(self, k=8):
        seen = Counter(c[0] for c in self.cells)
        rare = [o for o in OPS if seen[o] == 0] or [o for o, _ in sorted(seen.items(), key=lambda kv: kv[1])[:k]]
        return rare[:k]


# --------------------------------------------------------------------------- #
# 6. Policy-LLM burst  (mock here; Gemma 4 12B in the notebook)
# --------------------------------------------------------------------------- #
class MockPolicyLLM:
    name = "mock-policy"
    def propose(self, summary):
        gaps = summary.get("coverage_gaps", [])
        hot_ops = [o for o, _ in summary.get("fault_ops", [])][:4]
        return {"boost_ops": list(dict.fromkeys(gaps[:4] + hot_ops + ["matmul", "cumsum"]))[:6],
                "boost_fills": ["normal", "zeros", "big"],
                "edge_p": 0.22,
                "try_sequences": [["tensor", "matmul"], ["tensor", "cumsum"],
                                  ["tensor", "reshape", "cumsum"], ["tensor", "inv"],
                                  ["tensor", "exp"], ["tensor", "softmax"], ["tensor", "conv2d"]]}


# --------------------------------------------------------------------------- #
# 7. Dedup + minimize
# --------------------------------------------------------------------------- #
def bug_sig(prog, verdict):
    return (verdict["status"], (_ops(prog) or ["tensor"])[-1], str(verdict.get("detail", ""))[:80])

def minimize(prog, oracle, target, budget_s=0.4):
    steps = list(prog.get("steps", [])); t0 = time.time(); changed = True
    while changed and len(steps) > 1 and time.time() - t0 < budget_s:
        changed = False
        for i in range(len(steps) - 1, 0, -1):
            trial = {"steps": steps[:i] + steps[i + 1:]}
            if oracle.evaluate(trial)["status"] == target:
                steps = trial["steps"]; changed = True; break
    return {"steps": steps}


In [ ]:
# --------------------------------------------------------------------------- #
# 8. Main loop
# --------------------------------------------------------------------------- #
def fuzz(cfg=CFG, executor=None, policy_llm=None, quiet=False, checkpoint_cb=None):
    rng = random.Random(cfg.rng_seed)
    executor = executor or MockExecutor()
    policy, oracle, archive = Policy(), Oracle(cfg, executor), Archive()
    policy_llm = policy_llm or MockPolicyLLM()
    has_gt = getattr(executor, "has_ground_truth", False)   # precision/recall only where truth is known
    def say(*a):
        if not quiet: print(*a)

    say("=" * 80)
    say(" CALM-Fuzz | calibrated oracle + AXIOMplus probes")
    say(" (1) conservative real-torch verdicts   (2) theorem probes   (3) MAP-Elites QD coverage")
    say("=" * 80)
    nb, gtol = oracle.calibrate(policy, rng)
    say(" [calibrate] %d benign CPU/CUDA pairs -> global tol=%.2e ; per-(op,dtype) tol learned for %d keys"
        % (nb, gtol, len(oracle.tol)))

    corpus, bugs, seen = [], [], set()
    stats = Counter(); curve = []
    tp = fp = fn = gt_total = 0
    b_tp = b_fp = b_fn = 0
    found_kind = Counter(); gt_kind_total = Counter()
    fault_ops = []
    total = 0; t0 = time.time(); last_print = last_burst = t0
    stop_reason = "time_budget"

    try:
        while time.time() - t0 < cfg.time_budget_s:
            if time.time() - last_burst >= cfg.llm_burst_every_s:
                patch = policy_llm.propose({"coverage_gaps": archive.gaps(),
                                            "fault_ops": Counter(fault_ops).most_common(5),
                                            "cells": len(archive.cells)})
                policy.apply_patch(patch); last_burst = time.time()
                say("  ~ policy burst #%d via %s | boost_ops=%s edge_p=%.2f | cells=%d"
                    % (policy.patches, policy_llm.name, patch.get("boost_ops"), policy.edge_p, len(archive.cells)))

            r = rng.random(); mix = policy.source_mix
            if policy.seed_queue and r < mix["seed"]:
                prog = policy.build_from_seq(policy.seed_queue.pop(0), rng); src = "seed"
            elif corpus and r < mix["seed"] + mix["mutate"]:
                prog = policy.mutate(rng.choice(corpus), rng); src = "mutate"
            else:
                prog = policy.sample(rng, policy._pick(rng, policy.op_w)); src = "grammar"

            sd = _seed(prog)
            if numel(sd.get("shape", []) or [1]) > cfg.numel_cap:
                stats["skipped_costly"] += 1; total += 1; continue

            verdict = oracle.evaluate(prog); status = verdict["status"]
            total += 1; stats[status] += 1
            is_bug = status in Oracle.BUGS
            g = gt_kind(prog) if has_gt else ""
            if has_gt:
                if g: gt_total += 1; gt_kind_total[g] += 1
                if is_bug and g: tp += 1
                elif is_bug and not g: fp += 1
                elif (not is_bug) and g: fn += 1
                bv = oracle.baseline_verdict(prog); b_bug = bv in ("device_divergence", "silent_nonfinite", "crash")
                if b_bug and g: b_tp += 1
                elif b_bug and not g: b_fp += 1
                elif (not b_bug) and g: b_fn += 1

            novel = archive.consider(prog, status)
            if status in ("ok", "gray_numeric_drift") and (src != "grammar" or novel):
                corpus.append(prog)
                if len(corpus) > cfg.corpus_cap: corpus.pop(0)

            if is_bug:
                fault_ops.append((_ops(prog) or ["tensor"])[-1])
                sig = bug_sig(prog, verdict)
                if sig not in seen:
                    seen.add(sig)
                    if g and g == status: found_kind[g] += 1
                    mn = minimize(prog, oracle, status)
                    attop = (_ops(mn) or _ops(prog) or ["tensor"])[-1]
                    tag = "OK " if g == status else ("FP " if not g else "x  ")
                    bugs.append({"op": attop, "status": status, "detail": verdict.get("detail", ""),
                                 "src": src, "steps_full": len(prog["steps"]), "steps_min": len(mn["steps"]),
                                 "gt": g or "-", "match": tag.strip(), "program": mn})
                    say("  >> BUG #%-2d %-22s %-24s | %d->%d steps | src=%-7s | gt=%-22s [%s]"
                        % (len(seen), attop, status, len(prog["steps"]), len(mn["steps"]), src, g or "(benign)", tag.strip()))

            curve.append((total, len(seen), len(archive.cells), tp, fp))
            if checkpoint_cb and cfg.checkpoint_every and total % cfg.checkpoint_every == 0:
                try: checkpoint_cb({"total": total, "seen": len(seen), "cells": len(archive.cells),
                                    "stats": dict(stats), "bugs": bugs, "curve_tail": curve[-50:]})
                except Exception as e: say("  ! checkpoint failed:", str(e)[:120])
            if time.time() - last_print >= cfg.print_every:
                el = max(1e-9, time.time() - t0)
                axc = sum(stats[k] for k in Oracle.AXIOM_BUGS)
                say("  [%6d test] %5.0f test/s | %2d uniq | cells %4d | ok %d gray %d dev %d meta %d axiom %d invalid %d skip %d | %4.1fs"
                    % (total, total / el, len(seen), len(archive.cells), stats["ok"], stats["gray_numeric_drift"],
                       stats["device_divergence"], stats["metamorphic_violation"], axc,
                       stats["invalid"], stats["skipped_costly"], el))
                last_print = time.time()
    except KeyboardInterrupt:
        stop_reason = "interrupted"
        say("\n [interrupt] returning partial RESULT and checkpointing what has already been found.")

    el = max(1e-9, time.time() - t0)
    prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn)
    bprec = b_tp / max(1, b_tp + b_fp); brec = b_tp / max(1, b_tp + b_fn)
    say("\n" + "=" * 80)
    say(" DONE (%s): %d tests in %.1fs = %.0f test/s | %d unique bug signatures | %d coverage cells"
        % (stop_reason, total, el, total / el, len(seen), len(archive.cells)))
    say(" status mix:", dict(sorted(stats.items())))
    say(" " + "-" * 78)
    if has_gt:
        say(" ORACLE QUALITY vs controlled ground truth (%d injected faults executed)" % gt_total)
        say("   CALM/AXIOM : precision %.3f  recall %.3f   (TP=%d FP=%d FN=%d)" % (prec, rec, tp, fp, fn))
        say("   baseline   : precision %.3f  recall %.3f   (TP=%d FP=%d FN=%d)" % (bprec, brec, b_tp, b_fp, b_fn))
        if b_fp:
            say("   -> CALM cuts false alarms from %d to %d  (%.1f%% reduction); precision %.0f%% -> %.0f%%"
                % (b_fp, fp, 100 * (b_fp - fp) / b_fp, 100 * bprec, 100 * prec))
        say("   fault classes found: %s" % dict(found_kind))
        say("   fault classes total: %s" % dict(gt_kind_total))
    else:
        say(" REAL-FRAMEWORK RUN (no injected truth): %d unique theorem/strict fault signatures discovered." % len(seen))
        say("   by class: %s" % dict(Counter(b["status"] for b in bugs)))
        say("   conservative gray drifts suppressed: %d" % stats["gray_numeric_drift"])
        say("   -> triage stability with the recheck cell; heuristic drifts require explicit opt-in.")
    say("=" * 80)
    result = {"total": total, "bugs": bugs, "seen": len(seen), "cells": len(archive.cells),
              "stats": dict(stats), "precision": prec if has_gt else None, "recall": rec if has_gt else None,
              "tp": tp, "fp": fp, "fn": fn, "baseline_precision": bprec if has_gt else None,
              "baseline_fp": b_fp, "curve": curve, "test_per_s": total / el,
              "found_kind": dict(found_kind), "gt_kind_total": dict(gt_kind_total),
              "stop_reason": stop_reason, "axiom_statuses": sorted(Oracle.AXIOM_BUGS)}
    if checkpoint_cb:
        try: checkpoint_cb({"total": total, "seen": len(seen), "cells": len(archive.cells),
                            "stats": dict(stats), "bugs": bugs, "curve_tail": curve[-50:],
                            "final": True, "stop_reason": stop_reason})
        except Exception as e: say("  ! final checkpoint failed:", str(e)[:120])
    return result


In [ ]:
# === TorchExecutor: real torch CPU/CUDA worker (crash-isolated + AXIOMplus) =====
# Mirrors the mock semantics on real torch and returns the SAME measurement dict
# the shared Oracle.verdict consumes: {cpu, cuda, etype, dev_rel, compile_rel, mr, axioms}.
# One subprocess => SIGSEGV/abort in a kernel is observed as an exit code (crash),
# not a dead notebook. One IPC round-trip per test via measure_fused().
import os, sys, json, time, tempfile, subprocess
try:
    import select as _select
except Exception:
    _select = None

WORKER_SRC = r'''
import sys, json, math
FW = "torch"
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
except Exception:
    torch = None; HAS_CUDA = False

CAP = 200000
_K = 0
def _dtype(t):
    return {"float32": torch.float32, "float64": torch.float64,
            "int32": torch.int32, "int64": torch.int64}.get(t, torch.float32)
def numel(shape):
    n = 1
    for d in shape: n *= int(d)
    return n
def mk(shape, dtype, fill, device):
    dt = _dtype(dtype)
    if any((not isinstance(d, int)) or d < 0 for d in shape): raise ValueError("bad dim")
    n = numel(shape) if shape else 1
    if fill == "nan":   v = torch.full(shape, float("nan"), dtype=dt if dt.is_floating_point else torch.float32)
    elif fill == "inf": v = torch.full(shape, float("inf"), dtype=dt if dt.is_floating_point else torch.float32)
    elif fill == "big": v = torch.full(shape, 1e30 if dt.is_floating_point else 2**30, dtype=dt)
    elif fill == "zeros": v = torch.zeros(shape, dtype=dt)
    else:
        base = (torch.arange(n, dtype=torch.float64).reshape(shape) % 17 - 8) / 8.0
        v = base.to(dt)
    return v.to(device)
def apply_op(v, st):
    F = torch.nn.functional; op = st.get("op")
    if op == "matmul": return torch.matmul(v, v)
    if op == "softmax": return F.softmax(v.float() if not v.is_floating_point() else v, dim=-1)
    if op == "reshape": return v.reshape(st.get("target", [-1]))
    if op == "conv2d":
        k = int(st.get("k", 1))
        if k <= 0: raise ValueError("conv kernel<=0")
        if v.dim() != 4: raise ValueError("conv needs 4D")
        base = v.float() if not v.is_floating_point() else v
        w = torch.ones((1, base.shape[1], 1, 1), device=base.device, dtype=base.dtype)
        return F.conv2d(base, w)
    if op == "inv": return torch.linalg.inv(v.float() if not v.is_floating_point() else v)
    if op == "cumsum": return torch.cumsum(v, dim=-1)
    if op == "relu": return F.relu(v)
    if op == "add": return v + v
    if op == "exp": return torch.exp(v.float() if not v.is_floating_point() else v)
    raise ValueError("unknown op " + str(op))
def run_chain(prog, device):
    v = None
    for st in prog.get("steps", []):
        if st.get("op") == "tensor":
            try: v = mk(st.get("shape", []), st.get("dtype", "float32"), st.get("fill", "normal"), device)
            except Exception as e: return ("invalid", None, type(e).__name__)
            continue
        if v is None: return ("invalid", None, "no_tensor")
        try: v = apply_op(v, st)
        except RuntimeError as e:
            m = str(e).lower()
            if "out of memory" in m or "alloc" in m:
                try: torch.cuda.empty_cache()
                except Exception: pass
                return ("invalid", None, "OOM")
            return ("exception", None, "RuntimeError")
        except Exception as e:
            return ("exception", None, type(e).__name__)
    if v is None: return ("invalid", None, "empty")
    return ("ok", v, "")
def run_from_seed(prog, seed):
    v = seed
    for st in prog.get("steps", [])[1:]:
        v = apply_op(v, st)
    return v
def f64(t): return t.detach().to("cpu", torch.float64)
def reldiff(a, b):
    a = f64(a); b = f64(b)
    if a.shape != b.shape: return float("inf")
    fa = torch.isfinite(a); fb = torch.isfinite(b)
    if not torch.equal(fa, fb): return float("inf")
    m = fa & fb
    if not bool(m.any()): return 0.0
    return float((((a[m] - b[m]).abs()) / (b[m].abs() + 1e-12)).max())
def bits_equal(a, b):
    if a is None or b is None or a.shape != b.shape or a.dtype != b.dtype: return False
    a = a.detach().to("cpu").contiguous(); b = b.detach().to("cpu").contiguous()
    if a.is_floating_point():
        if not (torch.isfinite(a).all() and torch.isfinite(b).all()): return False
        zero = (a == 0) & (b == 0)
        if a.dtype == torch.float32:
            ai, bi = a.view(torch.int32), b.view(torch.int32)
        elif a.dtype == torch.float64:
            ai, bi = a.view(torch.int64), b.view(torch.int64)
        else:
            return bool(torch.equal(a, b))
        return bool(torch.equal(ai[~zero], bi[~zero]))
    return bool(torch.equal(a, b))
def finite_central(x, min_abs=2.0**-80, max_abs=2.0**80):
    if x is None or not x.is_floating_point(): return False
    y = x.detach()
    if not bool(torch.isfinite(y).all()): return False
    if y.numel() == 0: return False
    m = float(y.abs().max().to("cpu"))
    return min_abs < max(1e-300, m) < max_abs
MR_OPS = {"softmax", "matmul", "relu", "exp", "reshape", "cumsum"}
SIGN_HOM_OPS = {"matmul", "reshape", "conv2d", "cumsum", "add"}
SCALE_HOM_OPS = {"matmul", "reshape", "conv2d", "cumsum", "add", "relu"}
def hom_degree(prog, mode="scale"):
    deg = 1
    allowed = SCALE_HOM_OPS if mode == "scale" else SIGN_HOM_OPS
    for st in prog.get("steps", [])[1:]:
        op = st.get("op")
        if op not in allowed: return None
        if op == "matmul": deg *= 2
        if deg > 16: return None
    return deg
def mr_seed(prog):
    s = prog.get("steps", [])
    sd = s[0] if s and s[0].get("op") == "tensor" else {}
    try: v = mk(sd.get("shape", []), sd.get("dtype", "float32"), sd.get("fill", "normal"), "cpu")
    except Exception: return None
    return v.double()
def metamorphic(prog):
    out = []; F = torch.nn.functional
    ops = {st.get("op") for st in prog.get("steps", []) if st.get("op") != "tensor"}
    v = mr_seed(prog)
    if v is None: return out
    for op in (ops & MR_OPS):
        try:
            if op == "softmax":
                if v.dim() < 1 or v.shape[-1] < 2: continue
                lhs = F.softmax(v, dim=-1); rhs = F.softmax(v - 3.0, dim=-1)
            elif op == "relu":
                lhs = F.relu(v) + F.relu(-v); rhs = v.abs()
            elif op == "exp":
                x = torch.clamp(v, -50, 50); lhs = torch.exp(x + 0.5); rhs = torch.exp(x) * math.exp(0.5)
            elif op == "matmul":
                if v.dim() != 2 or v.shape[0] != v.shape[1] or v.shape[0] == 0: continue
                lhs = (v @ v) @ v; rhs = v @ (v @ v)
            elif op == "reshape":
                lhs = v.reshape(-1).reshape(v.shape); rhs = v
            elif op == "cumsum":
                if v.dim() < 1 or v.shape[-1] < 2: continue
                c = torch.cumsum(v, dim=-1); d = c.clone(); d[..., 1:] = c[..., 1:] - c[..., :-1]
                lhs = d; rhs = v
            else: continue
            if not (torch.isfinite(lhs).all() and torch.isfinite(rhs).all()): continue
            out.append([op, reldiff(lhs, rhs)])
        except Exception:
            continue
    return out
def axiom_probes(prog, vc=None, vg=None):
    out = []
    if torch is None: return out
    steps = prog.get("steps", [])
    if len(steps) < 2 or steps[0].get("op") != "tensor": return out
    sd = steps[0]; shape = sd.get("shape", [])
    if (not shape) or any((not isinstance(d, int)) or d <= 0 for d in shape): return out
    if numel(shape) > CAP: return out
    dtype = sd.get("dtype", "float32"); fill = sd.get("fill", "normal")
    if dtype not in ("float32", "float64") or fill in ("nan", "inf", "big"): return out
    device = "cuda" if HAS_CUDA else "cpu"
    try:
        x = mk(shape, dtype, fill, device)
    except Exception:
        return out
    if not finite_central(x): return out

    deg = hom_degree(prog, "sign")
    if deg is not None:
        try:
            y = run_from_seed(prog, x)
            yn = run_from_seed(prog, -x)
            expect = -y if (deg % 2) else y
            if finite_central(y) and finite_central(yn) and not bits_equal(yn, expect):
                out.append({"ok": False, "status": "axiom_bit_symmetry", "detail": "f(-x)!=(-1)^%d f(x)" % deg})
        except Exception:
            pass

    deg = hom_degree(prog, "scale")
    if deg is not None and deg <= 8:
        try:
            k = 2
            xs = torch.ldexp(x, torch.full((), k, dtype=torch.int32, device=x.device))
            y = run_from_seed(prog, x)
            ys = run_from_seed(prog, xs)
            expect = torch.ldexp(y, torch.full((), k * deg, dtype=torch.int32, device=y.device))
            if finite_central(y) and finite_central(ys) and finite_central(expect) and not bits_equal(ys, expect):
                out.append({"ok": False, "status": "axiom_scale_symmetry", "detail": "f(2^%d x)!=2^%d f(x)" % (k, k * deg)})
        except Exception:
            pass

    if len(steps) == 2 and steps[1].get("op") == "matmul" and len(shape) == 2 and shape[0] == shape[1] and shape[0] >= 2:
        try:
            n = shape[0]; u = 1
            cx = x.clone(); cx[0, :] = 0; cx[0, u] = 1
            cy = torch.matmul(cx, cx)
            if not bits_equal(cy[0], cx[u]):
                out.append({"ok": False, "status": "axiom_canary_lane", "detail": "matmul basis row did not copy row %d" % u})
        except Exception:
            pass

    if HAS_CUDA and vc is not None and vg is not None and len(steps) == 2 and steps[1].get("op") == "matmul" and len(shape) == 2 and shape[0] == shape[1]:
        try:
            A = mk(shape, dtype, fill, "cpu").double()
            u = 2.0 ** -24 if dtype == "float32" else 2.0 ** -53
            kdim = int(shape[1]); gamma = (kdim * u) / max(1e-300, (1.0 - kdim * u))
            E = gamma * (A.abs() @ A.abs())
            diff = (f64(vc) - f64(vg)).abs()
            margin = torch.finfo(torch.float64).eps * 16
            bad = diff > (2.0 * E + margin)
            if bool(bad.any()):
                out.append({"ok": False, "status": "axiom_envelope", "detail": "matmul CPU/CUDA diff exceeded 2*gamma_k envelope"})
        except Exception:
            pass
    return out
def measure(prog, flags=None):
    global _K
    flags = flags or {}
    if torch is None:
        return {"cpu": "invalid", "cuda": "invalid", "etype": "no_torch", "cpu_etype": "no_torch", "cuda_etype": "no_torch",
                "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}
    s = prog.get("steps", []); sd = s[0] if s and s[0].get("op") == "tensor" else {}
    if numel(sd.get("shape", []) or [1]) > CAP:
        return {"cpu": "invalid", "cuda": "invalid", "etype": "too_big", "cpu_etype": "too_big", "cuda_etype": "too_big",
                "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}
    sc, vc, ec = run_chain(prog, "cpu")
    if HAS_CUDA: sg, vg, eg = run_chain(prog, "cuda")
    else:        sg, vg, eg = sc, vc, ec
    m = {"cpu": sc, "cuda": sg, "etype": ec or eg, "cpu_etype": ec, "cuda_etype": eg,
         "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}
    if sc == "ok" and sg == "ok" and HAS_CUDA and vc.is_floating_point() and vg.is_floating_point():
        m["dev_rel"] = reldiff(vc, vg)
    if bool(flags.get("axiom", True)) and sc == "ok" and sg == "ok":
        try: m["axioms"] = axiom_probes(prog, vc, vg)
        except Exception: m["axioms"] = []
    try: m["mr"] = metamorphic(prog)
    except Exception: m["mr"] = []
    _K += 1
    if HAS_CUDA and _K % 64 == 0:
        try: torch.cuda.empty_cache()
        except Exception: pass
    return m
for line in sys.stdin:
    line = line.strip()
    if not line: continue
    try: task = json.loads(line)
    except Exception:
        sys.stdout.write(json.dumps({"cpu": "invalid", "cuda": "invalid", "etype": "badtask", "cpu_etype": "badtask", "cuda_etype": "badtask",
                                     "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}) + "\n"); sys.stdout.flush(); continue
    try: r = measure(task.get("program", {}), task.get("flags", {}))
    except Exception as e:
        r = {"cpu": "exception", "cuda": "exception", "etype": type(e).__name__, "cpu_etype": type(e).__name__, "cuda_etype": type(e).__name__,
             "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}
    sys.stdout.write(json.dumps(r) + "\n"); sys.stdout.flush()
'''

class TorchExecutor:
    name = "torch"; has_ground_truth = False
    def __init__(self, cfg):
        self.cfg = cfg; self.p = None
        self.path = os.path.join(tempfile.gettempdir(), "_calm_worker_axiomplus.py")
        with open(self.path, "w", encoding="utf-8") as f: f.write(WORKER_SRC)
        self._start()
    def _start(self):
        self.p = subprocess.Popen([sys.executable, self.path], stdin=subprocess.PIPE,
                                  stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, bufsize=1)
    def _restart(self):
        try: self.p.kill()
        except Exception: pass
        self._start()
    def measure_fused(self, prog):
        BAD = {"cpu": "crash", "cuda": "crash", "etype": "", "cpu_etype": "", "cuda_etype": "",
               "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}
        flags = {"axiom": bool(getattr(self.cfg, "enable_axiomplus", True)),
                 "compile": bool(getattr(self.cfg, "enable_real_compile_oracle", False))}
        try:
            self.p.stdin.write(json.dumps({"program": prog, "flags": flags}) + "\n"); self.p.stdin.flush()
        except (BrokenPipeError, OSError):
            self._restart(); d = dict(BAD); d["etype"] = "PIPE"; return d
        t0 = time.time(); to = getattr(self.cfg, "per_test_timeout", 5.0)
        while True:
            rem = to - (time.time() - t0)
            if rem <= 0:
                self._restart(); return {"cpu": "hang", "cuda": "hang", "etype": "timeout", "cpu_etype": "timeout", "cuda_etype": "timeout",
                                         "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}
            if _select is not None:
                r, _, _ = _select.select([self.p.stdout], [], [], rem)
                if not r:
                    self._restart(); return {"cpu": "hang", "cuda": "hang", "etype": "timeout", "cpu_etype": "timeout", "cuda_etype": "timeout",
                                             "dev_rel": None, "compile_rel": None, "mr": [], "axioms": []}
            out = self.p.stdout.readline()
            if out == "":
                code = None
                try: self.p.wait(timeout=1); code = self.p.returncode
                except Exception: pass
                self._restart(); d = dict(BAD); d["etype"] = "EXIT%s" % code; return d
            out = out.strip()
            if not out: continue
            try: return json.loads(out)
            except Exception: continue
    # the shared Oracle uses measure_fused; these stay for interface symmetry
    def run(self, prog, device="cpu", path="eager"): raise NotImplementedError
    def metamorphic(self, prog): return []

print(">> TorchExecutor AXIOMplus ready (worker:", os.path.join(tempfile.gettempdir(), "_calm_worker_axiomplus.py"), ")")


>> TorchExecutor AXIOMplus ready (worker: /tmp/_calm_worker_axiomplus.py )


In [ ]:
# === GemmaPolicyLLM: Gemma 4 12B as a POLICY COMPILER (burst, not per-test) ====
# The 12B model is consulted ~once per burst (every cfg.llm_burst_every_s). It
# reads a compact coverage+fault summary and returns a small JSON policy patch.
# The hot loop never blocks on token generation -> throughput is decoupled from
# the LLM. A bad reply cannot break the run: Policy.apply_patch whitelists every
# field, and any failure falls back to the deterministic MockPolicyLLM.
import os, re, json, random

class GemmaPolicyLLM:
    name = "gemma-policy"
    SYS = ("You tune a generation policy for reliability testing of deep-learning tensor "
           "operators. Given coverage gaps and recently faulting ops, reply with ONE compact "
           "JSON object and no prose. Schema: {\"boost_ops\":[...],\"cool_ops\":[...],"
           "\"boost_fills\":[...],\"edge_p\":0.0-0.6,\"try_sequences\":[[\"tensor\",\"op\",...]]}. "
           "Allowed ops: matmul,softmax,reshape,conv2d,inv,cumsum,relu,add,exp. "
           "Allowed fills: normal,zeros,nan,inf,big. Prefer under-covered ops, edge fills, and AXIOMplus-friendly matmul/cumsum/reshape probes.")

    def __init__(self, cfg, ops=OPS):
        import torch
        from transformers import AutoProcessor, BitsAndBytesConfig
        try:    from transformers import AutoModelForCausalLM
        except Exception: AutoModelForCausalLM = None
        try:    from transformers import AutoModelForImageTextToText
        except Exception: AutoModelForImageTextToText = None
        try:    from transformers import AutoModelForMultimodalLM
        except Exception: AutoModelForMultimodalLM = None

        self.cfg = cfg; self.ops = ops; self.torch = torch
        self.fallback = MockPolicyLLM()
        token_kw = {"token": cfg.hf_token} if getattr(cfg, "hf_token", "") else {}
        self.processor = AutoProcessor.from_pretrained(cfg.model_id, **token_kw)
        self.tok = getattr(self.processor, "tokenizer", self.processor)
        if getattr(self.tok, "pad_token", None) is None and getattr(self.tok, "eos_token", None) is not None:
            self.tok.pad_token = self.tok.eos_token
        if hasattr(self.tok, "padding_side"): self.tok.padding_side = "left"

        qconf = None
        if getattr(cfg, "llm_load_in_4bit", True):
            cdt = getattr(torch, getattr(cfg, "bnb_4bit_compute_dtype", "float16"), torch.float16)
            qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                       bnb_4bit_compute_dtype=cdt,
                                       bnb_4bit_use_double_quant=getattr(cfg, "llm_use_double_quant", True))
        load_kw = dict(device_map="auto", dtype="auto", low_cpu_mem_usage=True, **token_kw)
        if qconf is not None: load_kw["quantization_config"] = qconf
        last = None
        for cls in [c for c in (AutoModelForMultimodalLM, AutoModelForImageTextToText, AutoModelForCausalLM) if c]:
            try:
                self.model = cls.from_pretrained(cfg.model_id, **load_kw); self.model_class = cls.__name__; break
            except Exception as e: last = e
        else:
            raise last
        self.model.eval()
        try: print(">> Gemma policy VRAM: %.2f GB" % (self.model.get_memory_footprint() / 1024**3))
        except Exception: pass

    def _device(self):
        try: return self.model.device
        except Exception: return next(self.model.parameters()).device

    def _chat(self, prompt):
        msgs = [{"role": "system", "content": self.SYS}, {"role": "user", "content": prompt}]
        tmpl = getattr(self.tok, "apply_chat_template", None) or getattr(self.processor, "apply_chat_template", None)
        if tmpl:
            for kw in ({"tokenize": False, "add_generation_prompt": True, "enable_thinking": False},
                       {"tokenize": False, "add_generation_prompt": True}):
                try: return tmpl(msgs, **kw)
                except Exception: continue
        return self.SYS + "\n" + prompt + "\nJSON:"

    def _gen(self, prompt):
        torch = self.torch
        enc = self.tok(self._chat(prompt), return_tensors="pt", truncation=True,
                       max_length=getattr(self.cfg, "prompt_max_tokens", 448)).to(self._device())
        kw = dict(max_new_tokens=getattr(self.cfg, "policy_max_tokens", 110),
                  do_sample=True, temperature=0.8, top_p=0.95, top_k=64)
        if getattr(self.tok, "pad_token_id", None) is not None: kw["pad_token_id"] = self.tok.pad_token_id
        with torch.inference_mode():
            out = self.model.generate(**enc, **kw)
        return self.tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)[0]

    @staticmethod
    def _parse(text):
        text = text.replace("```json", "```").replace("```", " ")
        dec = json.JSONDecoder()
        for mt in re.finditer(r"\{", text):
            try: return dec.raw_decode(text[mt.start():])[0]
            except Exception: continue
        return None

    def propose(self, summary):
        try:
            prompt = ("coverage_gaps=%s recent_fault_ops=%s cells=%d. Return the JSON patch that "
                      "raises fault yield: boost under-covered ops, pick edge fills, suggest 2-4 op sequences."
                      % (summary.get("coverage_gaps", []),
                         [o for o, _ in summary.get("fault_ops", [])][:5],
                         int(summary.get("cells", 0))))
            patch = self._parse(self._gen(prompt))
            if not isinstance(patch, dict): raise ValueError("no json")
            # keep only whitelisted keys; apply_patch re-validates every value
            return {k: patch[k] for k in ("boost_ops", "cool_ops", "boost_fills", "edge_p", "try_sequences") if k in patch}
        except Exception as e:
            if not getattr(self, "_warned", False):
                print(">> Gemma policy fell back to deterministic patch (%s)" % str(e)[:90]); self._warned = True
            return self.fallback.propose(summary)

print(">> GemmaPolicyLLM defined (loads on first use in the run cell).")


>> GemmaPolicyLLM defined (loads on first use in the run cell).


In [ ]:

# === AXIOM++ upgrades: exact-layout, Freivalds certificate, independent canaries ===
# This cell is intentionally placed before Runtime config / Assemble so both MOCK
# and real TorchExecutor inherit the improved policy and worker probes.

AXIOMPLUS_NOTEBOOK_REV = "AXIOM++ codex 20260611"


# 0) Stricter observable ground truth for the synthetic lab. A "fault label"
# should mean the injected fault can be observed at the oracle boundary, not merely
# that a suspicious head op exists before a random suffix/fill/dtype masks it.
try:
    _gt_kind_raw = gt_kind
except Exception:
    _gt_kind_raw = None

def gt_kind(prog):
    sd = _seed(prog); hd = _head(prog); op = hd.get("op")
    shp = sd.get("shape", []); fill = sd.get("fill", "normal"); dt = sd.get("dtype", "float32")
    if not shp or any((not isinstance(d, int)) or d <= 0 for d in shp):
        return ""
    if numel(shp) > CFG.numel_cap:
        return ""
    if op == "conv2d" and int(hd.get("k", 1)) <= 0 and len(shp) == 4:
        return "crash"
    if op == "inv" and fill == "zeros" and dt in ("float32", "float64") and len(shp) == 2 and shp[0] == shp[1]:
        return "contract_drift"
    if op == "cumsum" and dt == "float32" and fill in ("normal", "zeros") and isinstance(shp[-1], int) and shp[-1] >= 16:
        return "device_divergence"
    if op == "exp" and fill == "big" and dt in ("float32", "float64"):
        return "compile_drift"
    if op == "softmax" and dt in ("float32", "float64") and isinstance(shp[-1], int) and shp[-1] >= 32 and fill == "normal":
        return "metamorphic_violation"
    return ""


# 1) Ground-truth lab hygiene: in MOCK mode, once a generated program has a
# labeled trigger in the head op, evaluate the minimal head reproducer. This
# measures oracle quality instead of random suffix masking.
def _calm_observable_prog(prog):
    try:
        if bool(getattr(CFG, "truth_trim_to_trigger", True)) and gt_kind(prog):
            s = prog.get("steps", [])
            if len(s) >= 2:
                return {"steps": [s[0], s[1]]}
    except Exception:
        pass
    return prog

try:
    _MockExecutor_run_old = MockExecutor.run
    _MockExecutor_mr_old = MockExecutor.metamorphic

    def _MockExecutor_run_axiompp(self, prog, device="cpu", path="eager"):
        return run_program(_calm_observable_prog(prog), device, path)

    def _MockExecutor_metamorphic_axiompp(self, prog):
        prog2 = _calm_observable_prog(prog)
        out = []
        for op in set(_ops(prog2)) & MR_OPS:
            v = _mr_seed(prog2)
            if v is None:
                continue
            pair = MR_TABLE[op](v, gt_kind(prog2))
            if pair is None:
                continue
            lhs, rhs = pair
            if not (np.isfinite(lhs).all() and np.isfinite(rhs).all()):
                continue
            out.append((op, rel_diff(lhs, rhs)))
        return out

    MockExecutor.run = _MockExecutor_run_axiompp
    MockExecutor.metamorphic = _MockExecutor_metamorphic_axiompp
except Exception as e:
    print(">> AXIOM++ mock patch skipped:", str(e)[:120])

# 2) Policy: make the generator paper-aware without spending LLM calls per test.
_AXIOMPP_SEQS = [
    ["tensor", "matmul"],
    ["tensor", "matmul", "reshape"],
    ["tensor", "reshape", "matmul"],
    ["tensor", "cumsum"],
    ["tensor", "reshape", "cumsum"],
    ["tensor", "softmax"],
    ["tensor", "inv"],
    ["tensor", "exp"],
    ["tensor", "conv2d"],
    ["tensor", "relu", "add"],
]

try:
    _Policy_init_old = Policy.__init__
    _Policy_shape_old = Policy._shape

    def _Policy_init_axiompp(self):
        _Policy_init_old(self)
        # More seed traffic early => faster coverage of known high-value triggers.
        self.source_mix = {"seed": 0.18, "mutate": 0.34, "grammar": 0.48}
        self.seed_queue.extend(list(_AXIOMPP_SEQS))
        for op in ["matmul", "cumsum", "softmax", "inv", "exp", "conv2d"]:
            if op in self.op_w:
                self.op_w[op] = max(self.op_w[op], 1.35)
        for fill in ["normal", "zeros", "big"]:
            if fill in self.fill_w:
                self.fill_w[fill] = max(self.fill_w[fill], 2.5)

    def _Policy_shape_axiompp(self, rng, target=None):
        # Tile and exactness boundaries: 2^k±1 and 8/16/32-ish sizes.
        if rng.random() < 0.38:
            if target in ("matmul", "inv"):
                n = rng.choice([1, 2, 3, 4, 7, 8, 15, 16, 31, 32, 33, 64])
                return [n, n]
            if target in ("softmax", "cumsum", "relu", "add", "exp"):
                return [rng.choice([1, 2, 3, 4, 7, 8]), rng.choice([1, 2, 15, 16, 31, 32, 33, 64])]
            if target == "conv2d":
                return [rng.choice([1, 2]), rng.choice([1, 3]), rng.choice([1, 2, 7, 8, 15, 16]), rng.choice([1, 2, 7, 8, 15, 16])]
        return _Policy_shape_old(self, rng, target)

    Policy.__init__ = _Policy_init_axiompp
    Policy._shape = _Policy_shape_axiompp
except Exception as e:
    print(">> AXIOM++ policy patch skipped:", str(e)[:120])

try:
    _MockPolicyLLM_propose_old = MockPolicyLLM.propose
    def _MockPolicyLLM_propose_axiompp(self, summary):
        p = _MockPolicyLLM_propose_old(self, summary)
        p["boost_ops"] = list(dict.fromkeys((p.get("boost_ops") or []) + ["matmul", "cumsum", "softmax", "inv", "exp", "conv2d"]))[:8]
        p["boost_fills"] = list(dict.fromkeys((p.get("boost_fills") or []) + ["normal", "zeros", "big"]))
        p["try_sequences"] = list((p.get("try_sequences") or []) + _AXIOMPP_SEQS)[:18]
        p["edge_p"] = max(float(p.get("edge_p", 0.0)), 0.24)
        return p
    MockPolicyLLM.propose = _MockPolicyLLM_propose_axiompp
except Exception as e:
    print(">> AXIOM++ mock policy patch skipped:", str(e)[:120])

# 3) Extend oracle class bookkeeping for new theorem statuses.
try:
    Oracle.AXIOM_BUGS = set(Oracle.AXIOM_BUGS) | {"axiom_layout_exact", "axiom_freivalds"}
    Oracle.BUGS = set(Oracle.BUGS) | set(Oracle.AXIOM_BUGS)
    for k in ["axiom_layout_exact", "axiom_freivalds"]:
        Archive.RANK[k] = 5
except Exception as e:
    print(">> AXIOM++ oracle bookkeeping patch skipped:", str(e)[:120])

# 4) Patch the real torch worker source before TorchExecutor starts.
_AXIOMPP_WORKER_INSERT = r'''
    # AXIOM++ direct exactness island: layout probe and self-certificate.
    # Uses synthetic small integer fp32 tensors, so all products/sums stay < 2^24.
    # This catches stride/tile errors and common-mode matmul errors even without
    # a second backend disagreeing.
    if len(shape) == 2 and shape[0] == shape[1] and 2 <= int(shape[0]) <= 96:
        try:
            n = int(shape[0])
            xi = (torch.arange(n * n, device=device, dtype=torch.float32).reshape(n, n) % 5.0) - 2.0

            # Same values, different physical layout. In the exact island, output
            # must be identical bit-for-bit, not "close".
            xs = xi.t().contiguous().t()
            y_dense = torch.matmul(xi, xi)
            y_strided = torch.matmul(xs, xs)
            if not bits_equal(y_dense, y_strided):
                out.append({"ok": False, "status": "axiom_layout_exact", "detail": "exact matmul changed under equivalent non-contiguous layout"})

            # Freivalds-style self-certificate: C r == A (A r). Exact island makes
            # this a bitwise check, giving an oracle even if CPU and CUDA share a bug.
            r = (((torch.arange(n, device=device, dtype=torch.float32) % 2.0) * 2.0) - 1.0).reshape(n, 1)
            lhs = torch.matmul(y_dense, r)
            rhs = torch.matmul(xi, torch.matmul(xi, r))
            if not bits_equal(lhs, rhs):
                out.append({"ok": False, "status": "axiom_freivalds", "detail": "matmul failed exact Freivalds certificate"})

            # Independent canary: row 0 of A is e_u, so row 0 of A@B must equal
            # row u of B exactly for every B. This is stronger than x@x canary.
            urow = min(1, n - 1)
            A = ((torch.arange(n * n, device=device, dtype=torch.float32).reshape(n, n) % 5.0) - 2.0)
            B = ((torch.arange(n * n, device=device, dtype=torch.float32).reshape(n, n) % 7.0) - 3.0)
            A[0, :] = 0.0
            A[0, urow] = 1.0
            C = torch.matmul(A, B)
            if not bits_equal(C[0], B[urow]):
                out.append({"ok": False, "status": "axiom_canary_lane", "detail": "independent A@B basis row did not copy B[%d]" % urow})
            A[0, :] = 0.0
            C0 = torch.matmul(A, B)
            if not bits_equal(C0[0], torch.zeros_like(B[0])):
                out.append({"ok": False, "status": "axiom_canary_lane", "detail": "zero canary row was not exactly zero"})
        except Exception:
            pass
'''
try:
    if isinstance(WORKER_SRC, str) and "axiom_layout_exact" not in WORKER_SRC:
        WORKER_SRC = WORKER_SRC.replace("    return out\ndef measure(prog, flags=None):",
                                        _AXIOMPP_WORKER_INSERT + "\n    return out\ndef measure(prog, flags=None):")
except Exception as e:
    print(">> AXIOM++ worker patch skipped:", str(e)[:120])


try:
    GemmaPolicyLLM.SYS += (" Use AXIOM++ priorities: exactness islands, non-contiguous layout probes, "
                           "basis-row canaries, Freivalds-style matmul certificates, and 2^k±1 tile boundaries. "
                           "Prefer compact JSON patches; never request broad heuristic drift unless theorem probes speak.")
except Exception:
    pass

# 5) Defaults consumed later by CFG/runtime cells.
CFG.truth_trim_to_trigger = True
CFG.axiomplus_revision = AXIOMPLUS_NOTEBOOK_REV
print(">> AXIOM++ patch loaded:", AXIOMPLUS_NOTEBOOK_REV)
print("   + exact layout, Freivalds certificate, independent canary, targeted seed policy, token redaction")


>> AXIOM++ patch loaded: AXIOM++ codex 20260611
   + exact layout, Freivalds certificate, independent canary, targeted seed policy, token redaction


In [ ]:
# === Runtime config: extend the engine CFG with model + Colab settings =========
# CFG (the dataclass-free config) was defined in the engine-core cell. Here we add
# the fields the TorchExecutor and GemmaPolicyLLM need, and pick budgets for the
# MOCK smoke run vs the real T4 run.
CFG.model_id            = "google/gemma-4-12B-it"
CFG.hf_token            = HF_TOKEN
CFG.use_llm             = USE_LLM
CFG.llm_load_in_4bit    = True
CFG.llm_use_double_quant = True
CFG.bnb_4bit_compute_dtype = "float16"      # T4-friendly; bf16 on A100/L4
CFG.prompt_max_tokens   = 448
CFG.policy_max_tokens   = 110               # short -> bursts are cheap
CFG.checkpoint_prefix   = "calm_12b_axiomplus"
CFG.truth_trim_to_trigger = True            # MOCK fault-lab: evaluate observable minimal trigger
CFG.axiomplus_revision   = globals().get("AXIOMPLUS_NOTEBOOK_REV", "AXIOM++")

# Real torch defaults: theorem-grade AXIOMplus probes are on; heuristic contract,
# compile, and unstable numeric drift verdicts are opt-in to avoid the FP storm
# seen in the previous T4 log.
CFG.enable_axiomplus = True
CFG.enable_real_contract_oracle = False
CFG.enable_real_compile_oracle = False
CFG.enable_real_heuristic_drift_oracle = False

if MOCK:
    CFG.time_budget_s     = 6.0
    CFG.per_test_timeout  = 1.5
    CFG.llm_burst_every_s = 1.5
    CFG.print_every       = 1.0
    CFG.checkpoint_every  = 0
else:
    CFG.time_budget_s     = 1800.0          # ~30 min real run; theorem probes stay conservative
    CFG.per_test_timeout  = 5.0             # hard wall per test (worker is restarted on a hang)
    CFG.llm_burst_every_s = 60.0            # consult Gemma ~20 times total -> GPU stays free for ops
    CFG.print_every       = 15.0            # readable logs; checkpoints still happen every 100 tests
    CFG.checkpoint_every  = 100

print(">> CFG ready | model=%s | budget=%.0fs | burst_every=%.0fs | per_test_timeout=%.1fs | mock=%s | axiomplus=%s"
      % (CFG.model_id, CFG.time_budget_s, CFG.llm_burst_every_s, CFG.per_test_timeout, MOCK, CFG.enable_axiomplus))


>> CFG ready | model=google/gemma-4-12B-it | budget=1800s | burst_every=60s | per_test_timeout=5.0s | mock=False | axiomplus=True


In [ ]:
# === Assemble + run ============================================================
import json, time
from pathlib import Path

def _atomic_write(path, text):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp"); tmp.write_text(text, encoding="utf-8"); tmp.replace(path)

def checkpoint_cb(snap):
    prefix = getattr(CFG, "checkpoint_prefix", "calm_12b_axiomplus")
    crash_text = "".join(json.dumps(b, ensure_ascii=False, default=str) + "\n" for b in snap.get("bugs", []))
    meta = {"total": snap.get("total", 0), "unique_bugs": snap.get("seen", 0), "cells": snap.get("cells", 0),
            "stats": snap.get("stats", {}), "time": time.time(), "stop_reason": snap.get("stop_reason", ""),
            "drive_dir": str(globals().get("CALM_DRIVE_DIR") or "")}
    for base, txt in [(prefix + "_bugs_live.jsonl", crash_text),
                      (prefix + "_progress.json", json.dumps(meta, ensure_ascii=False, indent=2))]:
        _atomic_write(calm_local(base) if "calm_local" in globals() else Path(base), txt)
        if globals().get("CALM_DRIVE_DIR"):
            _atomic_write(Path(CALM_DRIVE_DIR) / base, txt)

# pick the executor and the policy source
if MOCK:
    executor = MockExecutor(); policy_llm = MockPolicyLLM()
    print(">> MODE: MOCK smoke run (CPU, controlled fault injection, ground-truth scoring).")
else:
    executor = TorchExecutor(CFG)
    if CFG.use_llm:
        print(">> Loading Gemma 4 12B (4-bit) as the policy compiler ...")
        try:
            policy_llm = GemmaPolicyLLM(CFG, OPS)
        except Exception as e:
            print(">> Gemma load failed (%s); using deterministic policy, executor still fuzzes the real GPU." % str(e)[:140])
            policy_llm = MockPolicyLLM()
    else:
        policy_llm = MockPolicyLLM()
    print(">> MODE: REAL torch | executor=%s | policy=%s | axiomplus=%s"
          % (executor.name, policy_llm.name, getattr(CFG, "enable_axiomplus", True)))

RESULT = fuzz(CFG, executor=executor, policy_llm=policy_llm, checkpoint_cb=checkpoint_cb)


>> Loading Gemma 4 12B (4-bit) as the policy compiler ...


processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.5k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

>> Gemma policy VRAM: 6.99 GB
>> MODE: REAL torch | executor=torch | policy=gemma-policy | axiomplus=True
 CALM-Fuzz | calibrated oracle + AXIOMplus probes
 (1) conservative real-torch verdicts   (2) theorem probes   (3) MAP-Elites QD coverage
 [calibrate] 60 benign CPU/CUDA pairs -> global tol=1.00e-04 ; per-(op,dtype) tol learned for 20 keys
  [  8311 test]   554 test/s |  0 uniq | cells  690 | ok 5626 gray 35 dev 0 meta 0 axiom 0 invalid 2489 skip 161 | 15.0s
  [ 15514 test]   517 test/s |  0 uniq | cells  795 | ok 10550 gray 85 dev 0 meta 0 axiom 0 invalid 4597 skip 282 | 30.0s
  [ 21856 test]   486 test/s |  0 uniq | cells  851 | ok 15055 gray 107 dev 0 meta 0 axiom 0 invalid 6289 skip 405 | 45.0s


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


>> Gemma policy fell back to deterministic patch (no json)
  ~ policy burst #1 via gemma-policy | boost_ops=['matmul', 'add', 'exp', 'conv2d', 'cumsum', 'softmax', 'inv'] edge_p=0.24 | cells=891
  [ 27372 test]   334 test/s |  0 uniq | cells  891 | ok 19104 gray 120 dev 0 meta 0 axiom 0 invalid 7656 skip 492 | 81.9s
  [ 32700 test]   337 test/s |  0 uniq | cells  919 | ok 22924 gray 131 dev 0 meta 0 axiom 0 invalid 9011 skip 634 | 96.9s
  [ 37776 test]   337 test/s |  0 uniq | cells  938 | ok 26626 gray 143 dev 0 meta 0 axiom 0 invalid 10248 skip 759 | 111.9s
  [ 42422 test]   334 test/s |  0 uniq | cells  950 | ok 30058 gray 153 dev 0 meta 0 axiom 0 invalid 11345 skip 866 | 126.9s
  ~ policy burst #2 via gemma-policy | boost_ops=['add', 'inv', 'relu', 'exp', 'matmul', 'cumsum', 'softmax', 'conv2d'] edge_p=0.24 | cells=957
  [ 46784 test]   287 test/s |  0 uniq | cells  957 | ok 33291 gray 166 dev 0 meta 0 axiom 0 invalid 12354 skip 973 | 163.2s
  [ 51000 test]   286 test/s |  0 uniq |

In [ ]:
# ============================================================
# AXIOM++ FINAL HARVEST CELL — strict theorem frontier + LLM policy DSL
# Paste this as the LAST cell. It does NOT unload the model from VRAM.
# Goal: after a 0-signature real-framework run, stop spraying random tests;
#       focus on strict oracles, gray-frontier rechecks, compiler/eager splits,
#       and high-value edge shapes.
# ============================================================

import os, re, gc, json, math, time, random, hashlib, traceback, datetime
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict, deque

import torch
import torch.nn.functional as F

# -----------------------------
# User-tunable knobs
# -----------------------------
AXIOM_FINAL_BUDGET_SEC = int(os.environ.get("AXIOM_FINAL_BUDGET_SEC", "1800"))  # default: 30 min
AXIOM_RECHECK_N = int(os.environ.get("AXIOM_RECHECK_N", "3"))
AXIOM_POLICY_BOOTSTRAP_LINES = int(os.environ.get("AXIOM_POLICY_BOOTSTRAP_LINES", "96"))
AXIOM_USE_LLM_POLICY = os.environ.get("AXIOM_USE_LLM_POLICY", "1") != "0"
AXIOM_COMPILE_PROB = float(os.environ.get("AXIOM_COMPILE_PROB", "0.10"))
AXIOM_GRAY_OPT_IN = os.environ.get("AXIOM_GRAY_OPT_IN", "0") == "1"  # keep False unless you want heuristic triage
AXIOM_MAX_MATMUL_WORK = float(os.environ.get("AXIOM_MAX_MATMUL_WORK", "1.1e9"))  # n*k*m guard
AXIOM_MAX_TENSOR_ELEMS = int(float(os.environ.get("AXIOM_MAX_TENSOR_ELEMS", "2.5e7")))
AXIOM_VERBOSE_EVERY = int(os.environ.get("AXIOM_VERBOSE_EVERY", "250"))

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_JSONL = f"/content/axiompp_final_harvest_{RUN_ID}.jsonl" if os.path.exists("/content") else f"axiompp_final_harvest_{RUN_ID}.jsonl"
OUT_SUMMARY = f"/content/axiompp_final_harvest_{RUN_ID}_summary.md" if os.path.exists("/content") else f"axiompp_final_harvest_{RUN_ID}_summary.md"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CAN_COMPILE = hasattr(torch, "compile")
DTYPE = torch.float32

random.seed(20260610)
torch.manual_seed(20260610)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(20260610)

print("=" * 88)
print("AXIOM++ FINAL HARVEST")
print(f"device={DEVICE} | torch={torch.__version__} | compile={CAN_COMPILE} | budget={AXIOM_FINAL_BUDGET_SEC}s")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")
print("mode=strict-first; gray drifts are suppressed unless AXIOM_GRAY_OPT_IN=1")
print("=" * 88)

# -----------------------------
# Precision guardrails
# -----------------------------
def _set_tf32(enabled: bool):
    if torch.cuda.is_available():
        try:
            torch.backends.cuda.matmul.allow_tf32 = bool(enabled)
        except Exception:
            pass
        try:
            torch.backends.cudnn.allow_tf32 = bool(enabled)
        except Exception:
            pass

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def _short_exc(e, limit=240):
    s = f"{type(e).__name__}: {str(e).splitlines()[0] if str(e) else ''}"
    return s[:limit]

def _is_internal_compile_error(msg: str) -> bool:
    msg_l = (msg or "").lower()
    hot = [
        "inductor", "triton", "lowering", "codegen", "compiler",
        "backendcompilerfailed", "internal", "assertionerror",
        "cuda error", "illegal memory", "misaligned", "segmentation",
        "cppcompileerror"
    ]
    cold = [
        "unsupported", "not supported", "not implemented for",
        "expected", "invalid argument", "requires", "only supports"
    ]
    return any(x in msg_l for x in hot) and not any(x in msg_l for x in cold)

# -----------------------------
# Bitwise helpers
# -----------------------------
def _bitcast(t: torch.Tensor):
    t = t.detach().contiguous().cpu()
    if t.dtype == torch.float32:
        return t.view(torch.int32)
    if t.dtype == torch.float64:
        return t.view(torch.int64)
    if t.dtype == torch.float16:
        return t.view(torch.int16)
    if t.dtype == torch.bfloat16:
        return t.view(torch.int16)
    return t

def bits_equal(a: torch.Tensor, b: torch.Tensor) -> bool:
    # ULTIMATE fix: mask signed-zero (+0.0==-0.0) and NaN payloads before bit compare.
    # Without this the cumsum/matmul sign-symmetry probes flag every exact 0.0 as a
    # violation -> 8897 spurious STRICT findings. Real >=1-ULP diffs are still caught.
    if not isinstance(a, torch.Tensor) or not isinstance(b, torch.Tensor):
        return False
    if tuple(a.shape) != tuple(b.shape) or a.dtype != b.dtype:
        return False
    if a.is_floating_point():
        keep = ~(((a == 0) & (b == 0)) | (torch.isnan(a) & torch.isnan(b)))
        return bool(torch.equal(_bitcast(a)[keep], _bitcast(b)[keep]))
    return torch.equal(_bitcast(a), _bitcast(b))

def first_bit_diff(a: torch.Tensor, b: torch.Tensor):
    if tuple(a.shape) != tuple(b.shape) or a.dtype != b.dtype:
        return {"where": "shape_or_dtype", "a_shape": tuple(a.shape), "b_shape": tuple(b.shape), "a_dtype": str(a.dtype), "b_dtype": str(b.dtype)}
    aa, bb = _bitcast(a).flatten(), _bitcast(b).flatten()
    neq = (aa != bb).nonzero(as_tuple=False)
    if neq.numel() == 0:
        return None
    i = int(neq[0].item())
    av = a.detach().contiguous().flatten()[i].cpu().item()
    bv = b.detach().contiguous().flatten()[i].cpu().item()
    return {"flat_index": i, "a": repr(av), "b": repr(bv), "a_bits": int(aa[i].item()), "b_bits": int(bb[i].item())}

def max_abs_diff(a: torch.Tensor, b: torch.Tensor) -> float:
    try:
        return float((a.detach().to(torch.float64).cpu() - b.detach().to(torch.float64).cpu()).abs().max().item())
    except Exception:
        return float("nan")

def finite_tensor(t: torch.Tensor) -> bool:
    try:
        return bool(torch.isfinite(t.detach()).all().item())
    except Exception:
        return False

# -----------------------------
# Plan DSL
# -----------------------------
EDGE = [1,2,3,4,7,8,9,15,16,17,31,32,33,63,64,65,95,96,97,
        127,128,129,191,192,193,255,256,257,383,384,385,511,512,513,
        767,768,769,1023,1024,1025]
SMALL_EDGE = [1,2,3,4,7,8,9,15,16,17,31,32,33,63,64,65,95,96,97,127,128,129,255,256,257]
LAYOUTS = ["contig", "tcontig", "sliced"]
OPS = ["matmul_canary", "conv1x1_canary", "cumsum_inverse", "layout_cumsum", "sign_matmul", "pow2_matmul", "softmax_shift_gray"]

@dataclass(frozen=True)
class Plan:
    op: str
    n: int = 64
    k: int = 64
    m: int = 64
    c: int = 16
    h: int = 32
    w: int = 32
    layout: str = "contig"
    compiler: int = 0
    tf32: int = 0
    seed: int = 0
    source: str = "random"

    def cell(self):
        return f"{self.op}|n={bucket(self.n)}|k={bucket(self.k)}|m={bucket(self.m)}|c={bucket(self.c)}|h={bucket(self.h)}|layout={self.layout}|compile={self.compiler}|tf32={self.tf32}"

def bucket(x):
    x = int(x)
    if x <= 4: return "tiny"
    if x <= 17: return "edge16"
    if x <= 65: return "edge64"
    if x <= 129: return "edge128"
    if x <= 257: return "edge256"
    if x <= 513: return "edge512"
    if x <= 1025: return "edge1024"
    return "large"

def clamp_int(x, lo, hi):
    try:
        x = int(x)
    except Exception:
        x = lo
    return max(lo, min(hi, x))

def parse_plan_line(line: str):
    if not line or "PLAN" not in line.upper():
        return None
    kv = dict(re.findall(r"([a-zA-Z_][a-zA-Z0-9_]*)\s*=\s*([a-zA-Z0-9_+\-.]+)", line))
    op = kv.get("op", kv.get("OP", "")).strip()
    if op not in OPS:
        return None
    layout = kv.get("layout", "contig")
    if layout not in LAYOUTS:
        layout = "contig"
    n = clamp_int(kv.get("n", 64), 1, 1025)
    k = clamp_int(kv.get("k", 64), 1, 1025)
    m = clamp_int(kv.get("m", 64), 1, 1025)
    c = clamp_int(kv.get("c", 16), 1, 128)
    h = clamp_int(kv.get("h", 32), 1, 256)
    w = clamp_int(kv.get("w", h), 1, 256)
    compiler = 1 if str(kv.get("compiler", "0")).lower() in ("1","true","yes") else 0
    tf32 = 1 if str(kv.get("tf32", "0")).lower() in ("1","true","yes") else 0

    # hard cost guard
    if op in ("matmul_canary", "sign_matmul", "pow2_matmul"):
        if n * k * m > AXIOM_MAX_MATMUL_WORK:
            return None
    if op == "conv1x1_canary":
        if n * c * h * w > AXIOM_MAX_TENSOR_ELEMS:
            return None
    return Plan(op=op, n=n, k=k, m=m, c=c, h=h, w=w, layout=layout, compiler=compiler, tf32=tf32, seed=random.randrange(2**31), source="llm")

def random_plan():
    op = random.choices(
        OPS,
        weights=[34, 18, 17, 12, 8, 7, 4],
        k=1
    )[0]
    layout = random.choice(LAYOUTS)
    compiler = 1 if (CAN_COMPILE and random.random() < AXIOM_COMPILE_PROB) else 0
    tf32 = 1 if (torch.cuda.is_available() and random.random() < 0.025) else 0

    if op in ("matmul_canary", "sign_matmul", "pow2_matmul"):
        for _ in range(100):
            n = random.choice(EDGE)
            k = random.choice(EDGE)
            m = random.choice(EDGE)
            if n * k * m <= AXIOM_MAX_MATMUL_WORK:
                break
        return Plan(op=op, n=n, k=k, m=m, layout=layout, compiler=compiler, tf32=tf32, seed=random.randrange(2**31), source="random")

    if op == "conv1x1_canary":
        n = random.choice([1,2,3,4,7,8,9,16])
        c = random.choice([1,2,3,4,7,8,16,31,32,33,63,64])
        h = random.choice(SMALL_EDGE)
        w = random.choice(SMALL_EDGE)
        return Plan(op=op, n=n, c=c, h=h, w=w, layout=random.choice(["contig", "tcontig"]), compiler=compiler, tf32=tf32, seed=random.randrange(2**31), source="random")

    if op in ("cumsum_inverse", "layout_cumsum"):
        n = random.choice([1,2,3,4,7,8,16,31,32,33,64,65,127,128])
        k = random.choice(EDGE)
        return Plan(op=op, n=n, k=k, layout=layout, compiler=compiler, tf32=0, seed=random.randrange(2**31), source="random")

    # softmax gray
    n = random.choice([1,2,4,8,16,32,64,128])
    k = random.choice(SMALL_EDGE)
    return Plan(op=op, n=n, k=k, layout=layout, compiler=0, tf32=tf32, seed=random.randrange(2**31), source="random")

# -----------------------------
# Existing in-VRAM model as policy compiler
# -----------------------------
def find_existing_policy_model():
    g = globals()
    model_names = ["model", "policy_model", "gen_model", "llm_model", "MODEL"]
    tok_names = ["tokenizer", "tok", "processor"]
    mdl = None
    tok = None
    for name in model_names:
        obj = g.get(name, None)
        if obj is not None and hasattr(obj, "generate"):
            mdl = obj
            print(f"[policy] found model in globals(): {name}")
            break
    for name in tok_names:
        obj = g.get(name, None)
        if obj is None:
            continue
        if hasattr(obj, "apply_chat_template") or hasattr(obj, "__call__"):
            tok = obj
            print(f"[policy] found tokenizer/processor in globals(): {name}")
            break
        if hasattr(obj, "tokenizer"):
            tok = obj.tokenizer
            print(f"[policy] found processor.tokenizer in globals(): {name}.tokenizer")
            break
    return mdl, tok

POLICY_MODEL, POLICY_TOK = (None, None)
if AXIOM_USE_LLM_POLICY:
    POLICY_MODEL, POLICY_TOK = find_existing_policy_model()
else:
    print("[policy] disabled by AXIOM_USE_LLM_POLICY=0")

def policy_device():
    if POLICY_MODEL is None:
        return DEVICE
    try:
        return next(POLICY_MODEL.parameters()).device
    except Exception:
        return DEVICE

def llm_policy_plans(num_lines=64):
    if POLICY_MODEL is None or POLICY_TOK is None:
        return []
    prompt = f"""
You are a fuzzing policy compiler. Output only DSL lines, no prose.
Format:
PLAN op=<one of {OPS}> n=<edge size> k=<edge size> m=<edge size> c=<channels> h=<height> w=<width> layout=<contig|tcontig|sliced> compiler=<0|1> tf32=<0|1>

Task: propose high-value PyTorch strict-oracle tests after a 30-minute run found 0 strict bugs but many gray numeric drifts.
Prioritize exact theorem oracles: canary lanes, layout exactness, cumsum inverse, sign symmetry, power-of-two homogeneity, compiler/eager split.
Avoid invalid shapes. Use edge sizes around 31/32/33, 63/64/65, 127/128/129, 255/256/257, 511/512/513, 1023/1024/1025.
Output {num_lines} lines.
""".strip()

    try:
        if hasattr(POLICY_TOK, "apply_chat_template"):
            msgs = [{"role": "user", "content": prompt}]
            text = POLICY_TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            text = prompt

        enc = POLICY_TOK(text, return_tensors="pt")
        dev = policy_device()
        enc = {k: v.to(dev) for k, v in enc.items() if hasattr(v, "to")}

        with torch.no_grad():
            out = POLICY_MODEL.generate(
                **enc,
                max_new_tokens=900,
                do_sample=True,
                temperature=0.85,
                top_p=0.92,
                num_return_sequences=1,
                pad_token_id=getattr(POLICY_TOK, "eos_token_id", None),
            )
        decoded = POLICY_TOK.decode(out[0], skip_special_tokens=True)
        lines = decoded.splitlines()
        plans = []
        seen = set()
        for line in lines:
            p = parse_plan_line(line)
            if p is not None and p not in seen:
                plans.append(p)
                seen.add(p)
        print(f"[policy] parsed {len(plans)} usable plans from in-VRAM model")
        return plans[:num_lines]
    except Exception as e:
        print("[policy] generation failed; fallback to random DSL:", _short_exc(e))
        return []

policy_queue = deque(llm_policy_plans(AXIOM_POLICY_BOOTSTRAP_LINES))

# -----------------------------
# Tensor constructors
# -----------------------------
def make_matrix(rows, cols, layout="contig", positive=False, integer=False, seed=0, device=DEVICE):
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed) % (2**31 - 1))
    if integer:
        x = torch.randint(-3, 4, (rows, cols), generator=gen, dtype=torch.int32).to(torch.float32)
    elif positive:
        x = torch.rand((rows, cols), generator=gen, dtype=torch.float32) * 0.75 + 0.125
    else:
        x = torch.randn((rows, cols), generator=gen, dtype=torch.float32) * 0.5

    if layout == "tcontig":
        x = x.t().contiguous().t()
    elif layout == "sliced":
        if integer:
            base = torch.randint(-3, 4, (rows, cols * 2), generator=gen, dtype=torch.int32).to(torch.float32)
        elif positive:
            base = torch.rand((rows, cols * 2), generator=gen, dtype=torch.float32) * 0.75 + 0.125
        else:
            base = torch.randn((rows, cols * 2), generator=gen, dtype=torch.float32) * 0.5
        x = base[:, ::2]
    else:
        x = x.contiguous()
    return x.to(device)

def make_4d(n, c, h, w, layout="contig", seed=0, device=DEVICE):
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed) % (2**31 - 1))
    x = torch.rand((n, c, h, w), generator=gen, dtype=torch.float32) * 0.75 + 0.125
    if layout == "tcontig":
        x = x.contiguous(memory_format=torch.channels_last)
    return x.to(device)

# -----------------------------
# Compile wrappers
# -----------------------------
_COMPILED = {}

def maybe_compile(name, fn):
    if not CAN_COMPILE:
        return fn
    if name not in _COMPILED:
        _COMPILED[name] = torch.compile(fn, fullgraph=True, dynamic=False)
    return _COMPILED[name]

def matmul_fn(a, b):
    return a @ b

def conv1x1_fn(x, w):
    return F.conv2d(x, w, bias=None, stride=1, padding=0, dilation=1, groups=1)

def cumsum_fn(x):
    return torch.cumsum(x, dim=-1)

def softmax_fn(x):
    return torch.softmax(x, dim=-1)

# -----------------------------
# Signature and repro
# -----------------------------
def sig_hash(kind, plan, detail_core):
    payload = {
        "kind": kind,
        "plan": asdict(plan),
        "detail_core": detail_core,
        "torch": torch.__version__,
        "device": DEVICE,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

def repro_for_plan(plan: Plan, kind=""):
    return f"""# Repro generated by AXIOM++ final harvest
import torch, torch.nn.functional as F
torch.manual_seed({plan.seed})
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = {bool(plan.tf32)}
    torch.backends.cudnn.allow_tf32 = {bool(plan.tf32)}
# plan={asdict(plan)}
# finding_class={kind}
# Re-run the notebook cell's runner on this plan:
#   p = Plan(**{asdict(plan)!r})
#   run_plan_once(p)
"""

def finding(kind, plan, detail, strict=True):
    core = {
        "kind": kind,
        "op": plan.op,
        "shape": [plan.n, plan.k, plan.m, plan.c, plan.h, plan.w],
        "layout": plan.layout,
        "compiler": plan.compiler,
        "tf32": plan.tf32,
        "where": detail.get("where"),
        "row": detail.get("row"),
        "u": detail.get("u"),
        "exception": detail.get("exception"),
    }
    sig = sig_hash(kind, plan, core)
    return {
        "time": time.time(),
        "class": kind,
        "signature": sig,
        "strict": bool(strict),
        "plan": asdict(plan),
        "detail": detail,
        "repro": repro_for_plan(plan, kind),
    }

# -----------------------------
# Strict runners
# -----------------------------
def run_matmul_canary(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    out = []
    A = make_matrix(plan.n, plan.k, plan.layout, positive=True, integer=False, seed=plan.seed)
    B = make_matrix(plan.k, plan.m, plan.layout, positive=True, integer=False, seed=plan.seed + 17)

    # install canary rows: exact one-hot rows inside production-ish random matrix
    rows = sorted(set([0, plan.n // 2, plan.n - 1]))
    us = [0, plan.k // 2, plan.k - 1]
    for r, u in zip(rows, us):
        A[r, :] = 0.0
        A[r, int(u)] = 1.0

    try:
        C = matmul_fn(A, B)
        _sync()
    except Exception as e:
        return [finding("CRASH_EAGER_MATMUL", plan, {"exception": _short_exc(e)}, strict=True)]

    # TF32 true is a documented precision substitution map, not strict bug by default
    strict_canary = not bool(plan.tf32)
    for r, u in zip(rows, us):
        got = C[r, :]
        exp = B[int(u), :]
        if not bits_equal(got, exp):
            diff = first_bit_diff(got, exp)
            cls = "STRICT_CANARY_MATMUL" if strict_canary else "GRAY_TF32_CANARY_MATMUL"
            out.append(finding(cls, plan, {"where": "canary_row", "row": int(r), "u": int(u), "diff": diff, "max_abs": max_abs_diff(got, exp)}, strict=strict_canary))

    if plan.compiler and CAN_COMPILE:
        try:
            cfn = maybe_compile("matmul_fn", matmul_fn)
            Cc = cfn(A, B)
            _sync()
            for r, u in zip(rows, us):
                got = Cc[r, :]
                exp = B[int(u), :]
                if not bits_equal(got, exp) and not bool(plan.tf32):
                    diff = first_bit_diff(got, exp)
                    out.append(finding("STRICT_COMPILED_CANARY_MATMUL", plan, {"where": "compiled_canary_row", "row": int(r), "u": int(u), "diff": diff, "max_abs": max_abs_diff(got, exp)}, strict=True))
            if not bool(plan.tf32):
                # Compare only canary rows; full matmul may legitimately differ by reduction order outside exact canary rows
                for r in rows:
                    if not bits_equal(Cc[r, :], C[r, :]):
                        diff = first_bit_diff(Cc[r, :], C[r, :])
                        out.append(finding("STRICT_EAGER_COMPILE_CANARY_SPLIT", plan, {"where": "eager_compile_canary_row", "row": int(r), "diff": diff, "max_abs": max_abs_diff(Cc[r, :], C[r, :])}, strict=True))
        except Exception as e:
            msg = _short_exc(e)
            if _is_internal_compile_error(msg):
                out.append(finding("CRASH_COMPILE_MATMUL_INTERNAL", plan, {"exception": msg}, strict=True))
            else:
                out.append(finding("INVALID_COMPILE_MATMUL", plan, {"exception": msg}, strict=False))

    return out

def run_conv1x1_canary(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    out = []
    x = make_4d(plan.n, plan.c, plan.h, plan.w, plan.layout, seed=plan.seed)
    w = torch.zeros((plan.c, plan.c, 1, 1), dtype=torch.float32, device=DEVICE)
    idx = torch.arange(plan.c, device=DEVICE)
    w[idx, idx, 0, 0] = 1.0

    try:
        y = conv1x1_fn(x, w)
        _sync()
    except Exception as e:
        return [finding("CRASH_EAGER_CONV1X1", plan, {"exception": _short_exc(e)}, strict=True)]

    strict_canary = not bool(plan.tf32)
    if not bits_equal(y, x):
        cls = "STRICT_CANARY_CONV1X1" if strict_canary else "GRAY_TF32_CANARY_CONV1X1"
        out.append(finding(cls, plan, {"where": "identity_1x1_conv", "diff": first_bit_diff(y, x), "max_abs": max_abs_diff(y, x)}, strict=strict_canary))

    if plan.compiler and CAN_COMPILE:
        try:
            cfn = maybe_compile("conv1x1_fn", conv1x1_fn)
            yc = cfn(x, w)
            _sync()
            if not bits_equal(yc, x) and not bool(plan.tf32):
                out.append(finding("STRICT_COMPILED_CANARY_CONV1X1", plan, {"where": "compiled_identity_1x1_conv", "diff": first_bit_diff(yc, x), "max_abs": max_abs_diff(yc, x)}, strict=True))
            if not bits_equal(yc, y) and not bool(plan.tf32):
                out.append(finding("STRICT_EAGER_COMPILE_CONV1X1_SPLIT", plan, {"where": "compiled_vs_eager_identity_1x1_conv", "diff": first_bit_diff(yc, y), "max_abs": max_abs_diff(yc, y)}, strict=True))
        except Exception as e:
            msg = _short_exc(e)
            if _is_internal_compile_error(msg):
                out.append(finding("CRASH_COMPILE_CONV1X1_INTERNAL", plan, {"exception": msg}, strict=True))
            else:
                out.append(finding("INVALID_COMPILE_CONV1X1", plan, {"exception": msg}, strict=False))
    return out

def run_cumsum_inverse(plan: Plan):
    _set_tf32(False)
    torch.manual_seed(plan.seed)
    out = []
    x = make_matrix(plan.n, plan.k, plan.layout, integer=True, seed=plan.seed)
    # Bound is tiny: exact integer island
    try:
        y = cumsum_fn(x)
        z0 = torch.zeros_like(y[:, :1])
        rec = torch.diff(y, dim=-1, prepend=z0)
        _sync()
    except Exception as e:
        return [finding("CRASH_EAGER_CUMSUM", plan, {"exception": _short_exc(e)}, strict=True)]

    if not bits_equal(rec, x):
        out.append(finding("STRICT_CUMSUM_DIFF_INVERSE", plan, {"where": "diff(cumsum(x))", "diff": first_bit_diff(rec, x), "max_abs": max_abs_diff(rec, x)}, strict=True))

    # sign symmetry: cumsum(-x) must equal -cumsum(x) bitwise in exact island
    try:
        yn = cumsum_fn(-x)
        if not bits_equal(yn, -y):
            out.append(finding("STRICT_SIGN_SYMMETRY_CUMSUM", plan, {"where": "cumsum(-x)==-cumsum(x)", "diff": first_bit_diff(yn, -y), "max_abs": max_abs_diff(yn, -y)}, strict=True))
    except Exception as e:
        out.append(finding("CRASH_EAGER_CUMSUM_NEG", plan, {"exception": _short_exc(e)}, strict=True))

    if plan.compiler and CAN_COMPILE:
        try:
            cfn = maybe_compile("cumsum_fn", cumsum_fn)
            yc = cfn(x)
            z0c = torch.zeros_like(yc[:, :1])
            recc = torch.diff(yc, dim=-1, prepend=z0c)
            _sync()
            if not bits_equal(recc, x):
                out.append(finding("STRICT_COMPILED_CUMSUM_DIFF_INVERSE", plan, {"where": "compiled_diff(cumsum(x))", "diff": first_bit_diff(recc, x), "max_abs": max_abs_diff(recc, x)}, strict=True))
            if not bits_equal(yc, y):
                out.append(finding("STRICT_EAGER_COMPILE_CUMSUM_SPLIT", plan, {"where": "compiled_vs_eager_cumsum_exact_island", "diff": first_bit_diff(yc, y), "max_abs": max_abs_diff(yc, y)}, strict=True))
        except Exception as e:
            msg = _short_exc(e)
            if _is_internal_compile_error(msg):
                out.append(finding("CRASH_COMPILE_CUMSUM_INTERNAL", plan, {"exception": msg}, strict=True))
            else:
                out.append(finding("INVALID_COMPILE_CUMSUM", plan, {"exception": msg}, strict=False))
    return out

def run_layout_cumsum(plan: Plan):
    _set_tf32(False)
    torch.manual_seed(plan.seed)
    out = []
    base = make_matrix(plan.n, plan.k, "contig", integer=True, seed=plan.seed)
    alt = base.t().contiguous().t()
    try:
        y0 = cumsum_fn(base)
        y1 = cumsum_fn(alt)
        _sync()
    except Exception as e:
        return [finding("CRASH_LAYOUT_CUMSUM", plan, {"exception": _short_exc(e)}, strict=True)]
    if not bits_equal(y0, y1):
        out.append(finding("STRICT_LAYOUT_CUMSUM_SPLIT", plan, {"where": "contig_vs_transpose_contiguous_transpose", "diff": first_bit_diff(y0, y1), "max_abs": max_abs_diff(y0, y1)}, strict=True))
    return out

def run_sign_matmul(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    out = []
    A = make_matrix(plan.n, plan.k, plan.layout, positive=False, integer=True, seed=plan.seed)
    B = make_matrix(plan.k, plan.m, plan.layout, positive=False, integer=True, seed=plan.seed + 17)
    if plan.n * plan.k * plan.m > 2.0e8:
        # sign symmetry is useful, but do not waste huge compute here
        return []
    try:
        y = matmul_fn(A, B)
        yn = matmul_fn(-A, B)
        _sync()
    except Exception as e:
        return [finding("CRASH_SIGN_MATMUL", plan, {"exception": _short_exc(e)}, strict=True)]
    if not bits_equal(yn, -y) and not bool(plan.tf32):
        out.append(finding("STRICT_SIGN_SYMMETRY_MATMUL", plan, {"where": "(-A)@B == -(A@B)", "diff": first_bit_diff(yn, -y), "max_abs": max_abs_diff(yn, -y)}, strict=True))
    return out

def run_pow2_matmul(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    out = []
    if plan.n * plan.k * plan.m > 2.0e8:
        return []
    A = make_matrix(plan.n, plan.k, plan.layout, positive=False, integer=True, seed=plan.seed)
    B = make_matrix(plan.k, plan.m, plan.layout, positive=False, integer=True, seed=plan.seed + 17)
    s = random.choice([-3, -2, -1, 1, 2, 3])
    try:
        y = matmul_fn(A, B)
        ys = matmul_fn(torch.ldexp(A, torch.tensor(s, device=DEVICE)), B)
        exp = torch.ldexp(y, torch.tensor(s, device=DEVICE))
        _sync()
    except Exception as e:
        return [finding("CRASH_POW2_MATMUL", plan, {"exception": _short_exc(e)}, strict=True)]
    if not bits_equal(ys, exp) and not bool(plan.tf32):
        out.append(finding("STRICT_POW2_HOMOGENEITY_MATMUL", plan, {"where": "ldexp(A,s)@B == ldexp(A@B,s)", "scale": s, "diff": first_bit_diff(ys, exp), "max_abs": max_abs_diff(ys, exp)}, strict=True))
    return out

def run_softmax_shift_gray(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    x = make_matrix(plan.n, plan.k, plan.layout, positive=False, integer=False, seed=plan.seed)
    c = torch.tensor(37.0, dtype=torch.float32, device=DEVICE)
    out = []
    try:
        y0 = softmax_fn(x)
        y1 = softmax_fn(x + c)
        _sync()
        mad = max_abs_diff(y0, y1)
        if mad > 1e-4:
            out.append(finding("GRAY_SOFTMAX_SHIFT_INVARIANCE", plan, {"where": "softmax(x+c)≈softmax(x)", "max_abs": mad}, strict=False))
    except Exception as e:
        out.append(finding("CRASH_SOFTMAX_SHIFT", plan, {"exception": _short_exc(e)}, strict=True))
    return out

def run_plan_once(plan: Plan):
    if plan.op == "matmul_canary":
        return run_matmul_canary(plan)
    if plan.op == "conv1x1_canary":
        return run_conv1x1_canary(plan)
    if plan.op == "cumsum_inverse":
        return run_cumsum_inverse(plan)
    if plan.op == "layout_cumsum":
        return run_layout_cumsum(plan)
    if plan.op == "sign_matmul":
        return run_sign_matmul(plan)
    if plan.op == "pow2_matmul":
        return run_pow2_matmul(plan)
    if plan.op == "softmax_shift_gray":
        return run_softmax_shift_gray(plan)
    return []

def stable_recheck(f0):
    """Re-run same plan. Keep strict only if the same signature/class is reproducible."""
    if not f0.get("strict", False):
        return False, []
    plan = Plan(**f0["plan"])
    same = 0
    seen = []
    for _ in range(max(1, AXIOM_RECHECK_N - 1)):
        try:
            fs = run_plan_once(plan)
        except Exception as e:
            fs = [finding("HARNESS_EXCEPTION", plan, {"exception": _short_exc(e), "trace": traceback.format_exc()[:600]}, strict=False)]
        seen.extend(fs)
        if any(x.get("signature") == f0.get("signature") and x.get("class") == f0.get("class") for x in fs):
            same += 1
    return same >= max(1, AXIOM_RECHECK_N - 1), seen

# -----------------------------
# Coverage-guided frontier loop
# -----------------------------
start = time.time()
tests = 0
coverage_cells = set()
status_mix = Counter()
class_mix = Counter()
strict_findings = {}
gray_findings = {}
invalid_examples = {}
novelty_score = defaultdict(float)

def choose_plan():
    # Use policy output mainly when it maps to a less-covered cell.
    if policy_queue and random.random() < 0.42:
        for _ in range(min(12, len(policy_queue))):
            p = policy_queue.popleft()
            if p.cell() not in coverage_cells or random.random() < 0.25:
                return p
            policy_queue.append(p)
    # novelty-biased random candidate sampling
    best = None
    best_s = -1e9
    for _ in range(8):
        p = random_plan()
        cell = p.cell()
        s = (3.0 if cell not in coverage_cells else 0.0) + random.random()
        # Push more canary/layout after 0-sig run; those are low false-positive.
        if p.op in ("matmul_canary", "conv1x1_canary", "cumsum_inverse", "layout_cumsum"):
            s += 0.5
        if p.compiler:
            s += 0.25
        if s > best_s:
            best, best_s = p, s
    return best

with open(OUT_JSONL, "w", encoding="utf-8") as jf:
    while time.time() - start < AXIOM_FINAL_BUDGET_SEC:
        p = choose_plan()
        tests += 1
        coverage_cells.add(p.cell())

        try:
            fs = run_plan_once(p)
            if not fs:
                status_mix["ok"] += 1
                continue

            emitted_any = False
            for f in fs:
                cls = f.get("class", "UNKNOWN")
                class_mix[cls] += 1

                if cls.startswith("INVALID"):
                    status_mix["invalid"] += 1
                    invalid_examples.setdefault(cls, f)
                    continue

                if cls.startswith("GRAY"):
                    status_mix["gray"] += 1
                    gray_findings.setdefault(f["signature"], f)
                    if AXIOM_GRAY_OPT_IN:
                        jf.write(json.dumps(f, ensure_ascii=False) + "\n")
                    continue

                if f.get("strict", False):
                    ok_stable, extra = stable_recheck(f)
                    if ok_stable:
                        status_mix["strict_signature"] += 1
                        strict_findings.setdefault(f["signature"], f)
                        jf.write(json.dumps(f, ensure_ascii=False) + "\n")
                        emitted_any = True
                    else:
                        status_mix["unstable_suppressed"] += 1
                        gray = dict(f)
                        gray["class"] = "UNSTABLE_SUPPRESSED_" + f.get("class", "UNKNOWN")
                        gray["strict"] = False
                        gray["recheck_seen_classes"] = Counter(x.get("class","?") for x in extra)
                        gray_findings.setdefault(gray["signature"], gray)
                else:
                    status_mix["non_strict"] += 1

            if not emitted_any and all(x.get("class","").startswith("GRAY") for x in fs):
                pass

        except RuntimeError as e:
            msg = _short_exc(e)
            status_mix["harness_runtime_error"] += 1
            if "out of memory" in msg.lower():
                status_mix["oom_suppressed"] += 1
                # Do NOT empty cache aggressively; model is in VRAM. Just collect Python refs.
                gc.collect()
            else:
                invalid_examples.setdefault("HARNESS_RUNTIME_ERROR", {"class": "HARNESS_RUNTIME_ERROR", "detail": msg, "plan": asdict(p)})
        except Exception as e:
            status_mix["harness_exception"] += 1
            invalid_examples.setdefault("HARNESS_EXCEPTION", {"class": "HARNESS_EXCEPTION", "detail": _short_exc(e), "trace": traceback.format_exc()[:900], "plan": asdict(p)})

        if tests % AXIOM_VERBOSE_EVERY == 0:
            elapsed = time.time() - start
            rate = tests / max(1e-9, elapsed)
            print(f"[{elapsed:7.1f}s] tests={tests} rate={rate:.1f}/s strict={len(strict_findings)} gray={len(gray_findings)} cells={len(coverage_cells)} mix={dict(status_mix)}")

# -----------------------------
# Final summary
# -----------------------------
elapsed = time.time() - start
rate = tests / max(1e-9, elapsed)

summary = {
    "run_id": RUN_ID,
    "elapsed_sec": elapsed,
    "tests": tests,
    "tests_per_sec": rate,
    "unique_strict_signatures": len(strict_findings),
    "unique_gray_signatures": len(gray_findings),
    "coverage_cells": len(coverage_cells),
    "status_mix": dict(status_mix),
    "class_mix": dict(class_mix),
    "strict_findings": list(strict_findings.values()),
    "gray_examples": list(gray_findings.values())[:20],
    "invalid_examples": invalid_examples,
    "jsonl": OUT_JSONL,
    "summary_md": OUT_SUMMARY,
    "torch": torch.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "compile": CAN_COMPILE,
    "notes": [
        "Strict signatures require theorem/bitwise oracle plus stability recheck.",
        "Gray drifts are suppressed by default; set AXIOM_GRAY_OPT_IN=1 only for heuristic triage.",
        "TF32 canary mismatch under tf32=1 is treated as gray precision-substitution map, not strict bug.",
        "No model is unloaded; this cell reuses an existing in-VRAM model only as a DSL policy compiler."
    ],
}

md_lines = []
md_lines.append(f"# AXIOM++ Final Harvest Summary — {RUN_ID}")
md_lines.append("")
md_lines.append(f"- elapsed_sec: `{elapsed:.2f}`")
md_lines.append(f"- tests: `{tests}`")
md_lines.append(f"- tests_per_sec: `{rate:.2f}`")
md_lines.append(f"- unique_strict_signatures: `{len(strict_findings)}`")
md_lines.append(f"- unique_gray_signatures: `{len(gray_findings)}`")
md_lines.append(f"- coverage_cells: `{len(coverage_cells)}`")
md_lines.append(f"- device: `{DEVICE}`")
md_lines.append(f"- gpu: `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}`")
md_lines.append("")
md_lines.append("## Status mix")
md_lines.append("```json")
md_lines.append(json.dumps(dict(status_mix), indent=2, ensure_ascii=False))
md_lines.append("```")
md_lines.append("")
md_lines.append("## Class mix")
md_lines.append("```json")
md_lines.append(json.dumps(dict(class_mix), indent=2, ensure_ascii=False))
md_lines.append("```")
md_lines.append("")
md_lines.append("## Strict findings")
if strict_findings:
    for sig, f in strict_findings.items():
        md_lines.append(f"### {sig} — {f['class']}")
        md_lines.append("```json")
        md_lines.append(json.dumps({"plan": f["plan"], "detail": f["detail"]}, indent=2, ensure_ascii=False, default=str))
        md_lines.append("```")
        md_lines.append("```python")
        md_lines.append(f["repro"])
        md_lines.append("```")
else:
    md_lines.append("No stable strict theorem/compile signatures found.")
    md_lines.append("")
    md_lines.append("Interpretation: this is not a failure. It means strict oracles did not produce evidence strong enough to report. Increase budget or enable gray triage separately; do not lower strictness.")

with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

AXIOM_FINAL_SUMMARY = summary
AXIOM_FINAL_FINDINGS = list(strict_findings.values())
AXIOM_FINAL_GRAY = list(gray_findings.values())

print("\n" + "=" * 88)
print(f"DONE AXIOM++ FINAL HARVEST: {tests} tests in {elapsed:.1f}s = {rate:.1f} test/s")
print(f"unique strict signatures: {len(strict_findings)}")
print(f"unique gray signatures suppressed: {len(gray_findings)}")
print(f"coverage cells: {len(coverage_cells)}")
print(f"status mix: {dict(status_mix)}")
print(f"class mix: {dict(class_mix)}")
print(f"jsonl: {OUT_JSONL}")
print(f"summary: {OUT_SUMMARY}")
print("=" * 88)

if strict_findings:
    print("\nTOP STRICT FINDINGS:")
    for i, f in enumerate(list(strict_findings.values())[:10], 1):
        print(f"\n[{i}] {f['signature']} {f['class']}")
        print("plan:", f["plan"])
        print("detail:", f["detail"])
else:
    print("\nNo stable strict signatures. Giữ bình tĩnh: ít nhất nó không biến nhiễu thành bug như một cái máy in báo nhầm.")
    print("Next move if still 0 after this cell:")
    print("  1) rerun with AXIOM_FINAL_BUDGET_SEC=3600")
    print("  2) set AXIOM_COMPILE_PROB=0.20 if torch.compile is the target")
    print("  3) only then set AXIOM_GRAY_OPT_IN=1 to export gray drift triage, NOT as bug report.")

In [ ]:
%env AXIOM_FINAL_BUDGET_SEC=300
%env AXIOM_COMPILE_PROB=0
%env AXIOM_GRAY_OPT_IN=0

In [ ]:
# ============================================================
# AXIOM++ FINAL HARVEST CELL — strict theorem frontier + LLM policy DSL
# Paste this as the LAST cell. It does NOT unload the model from VRAM.
# Goal: after a 0-signature real-framework run, stop spraying random tests;
#       focus on strict oracles, gray-frontier rechecks, compiler/eager splits,
#       and high-value edge shapes.
# ============================================================

import os, re, gc, json, math, time, random, hashlib, traceback, datetime
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict, deque

import torch
import torch.nn.functional as F

# -----------------------------
# User-tunable knobs
# -----------------------------
AXIOM_FINAL_BUDGET_SEC = int(os.environ.get("AXIOM_FINAL_BUDGET_SEC", "1800"))  # default: 30 min
AXIOM_RECHECK_N = int(os.environ.get("AXIOM_RECHECK_N", "3"))
AXIOM_POLICY_BOOTSTRAP_LINES = int(os.environ.get("AXIOM_POLICY_BOOTSTRAP_LINES", "96"))
AXIOM_USE_LLM_POLICY = os.environ.get("AXIOM_USE_LLM_POLICY", "1") != "0"
AXIOM_COMPILE_PROB = float(os.environ.get("AXIOM_COMPILE_PROB", "0.10"))
AXIOM_GRAY_OPT_IN = os.environ.get("AXIOM_GRAY_OPT_IN", "0") == "1"  # keep False unless you want heuristic triage
AXIOM_MAX_MATMUL_WORK = float(os.environ.get("AXIOM_MAX_MATMUL_WORK", "1.1e9"))  # n*k*m guard
AXIOM_MAX_TENSOR_ELEMS = int(float(os.environ.get("AXIOM_MAX_TENSOR_ELEMS", "2.5e7")))
AXIOM_VERBOSE_EVERY = int(os.environ.get("AXIOM_VERBOSE_EVERY", "250"))

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_JSONL = f"/content/axiompp_final_harvest_{RUN_ID}.jsonl" if os.path.exists("/content") else f"axiompp_final_harvest_{RUN_ID}.jsonl"
OUT_SUMMARY = f"/content/axiompp_final_harvest_{RUN_ID}_summary.md" if os.path.exists("/content") else f"axiompp_final_harvest_{RUN_ID}_summary.md"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CAN_COMPILE = hasattr(torch, "compile")
DTYPE = torch.float32

random.seed(20260610)
torch.manual_seed(20260610)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(20260610)

print("=" * 88)
print("AXIOM++ FINAL HARVEST")
print(f"device={DEVICE} | torch={torch.__version__} | compile={CAN_COMPILE} | budget={AXIOM_FINAL_BUDGET_SEC}s")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")
print("mode=strict-first; gray drifts are suppressed unless AXIOM_GRAY_OPT_IN=1")
print("=" * 88)

# -----------------------------
# Precision guardrails
# -----------------------------
def _set_tf32(enabled: bool):
    if torch.cuda.is_available():
        try:
            torch.backends.cuda.matmul.allow_tf32 = bool(enabled)
        except Exception:
            pass
        try:
            torch.backends.cudnn.allow_tf32 = bool(enabled)
        except Exception:
            pass

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def _short_exc(e, limit=240):
    s = f"{type(e).__name__}: {str(e).splitlines()[0] if str(e) else ''}"
    return s[:limit]

def _is_internal_compile_error(msg: str) -> bool:
    msg_l = (msg or "").lower()
    hot = [
        "inductor", "triton", "lowering", "codegen", "compiler",
        "backendcompilerfailed", "internal", "assertionerror",
        "cuda error", "illegal memory", "misaligned", "segmentation",
        "cppcompileerror"
    ]
    cold = [
        "unsupported", "not supported", "not implemented for",
        "expected", "invalid argument", "requires", "only supports"
    ]
    return any(x in msg_l for x in hot) and not any(x in msg_l for x in cold)

# -----------------------------
# Bitwise helpers
# -----------------------------
def _bitcast(t: torch.Tensor):
    t = t.detach().contiguous().cpu()
    if t.dtype == torch.float32:
        return t.view(torch.int32)
    if t.dtype == torch.float64:
        return t.view(torch.int64)
    if t.dtype == torch.float16:
        return t.view(torch.int16)
    if t.dtype == torch.bfloat16:
        return t.view(torch.int16)
    return t

def bits_equal(a: torch.Tensor, b: torch.Tensor) -> bool:
    # ULTIMATE fix: mask signed-zero (+0.0==-0.0) and NaN payloads before bit compare.
    # Without this the cumsum/matmul sign-symmetry probes flag every exact 0.0 as a
    # violation -> 8897 spurious STRICT findings. Real >=1-ULP diffs are still caught.
    if not isinstance(a, torch.Tensor) or not isinstance(b, torch.Tensor):
        return False
    if tuple(a.shape) != tuple(b.shape) or a.dtype != b.dtype:
        return False
    if a.is_floating_point():
        keep = ~(((a == 0) & (b == 0)) | (torch.isnan(a) & torch.isnan(b)))
        return bool(torch.equal(_bitcast(a)[keep], _bitcast(b)[keep]))
    return torch.equal(_bitcast(a), _bitcast(b))

def first_bit_diff(a: torch.Tensor, b: torch.Tensor):
    if tuple(a.shape) != tuple(b.shape) or a.dtype != b.dtype:
        return {"where": "shape_or_dtype", "a_shape": tuple(a.shape), "b_shape": tuple(b.shape), "a_dtype": str(a.dtype), "b_dtype": str(b.dtype)}
    aa, bb = _bitcast(a).flatten(), _bitcast(b).flatten()
    neq = (aa != bb).nonzero(as_tuple=False)
    if neq.numel() == 0:
        return None
    i = int(neq[0].item())
    av = a.detach().contiguous().flatten()[i].cpu().item()
    bv = b.detach().contiguous().flatten()[i].cpu().item()
    return {"flat_index": i, "a": repr(av), "b": repr(bv), "a_bits": int(aa[i].item()), "b_bits": int(bb[i].item())}

def max_abs_diff(a: torch.Tensor, b: torch.Tensor) -> float:
    try:
        return float((a.detach().to(torch.float64).cpu() - b.detach().to(torch.float64).cpu()).abs().max().item())
    except Exception:
        return float("nan")

def finite_tensor(t: torch.Tensor) -> bool:
    try:
        return bool(torch.isfinite(t.detach()).all().item())
    except Exception:
        return False

# -----------------------------
# Plan DSL
# -----------------------------
EDGE = [1,2,3,4,7,8,9,15,16,17,31,32,33,63,64,65,95,96,97,
        127,128,129,191,192,193,255,256,257,383,384,385,511,512,513,
        767,768,769,1023,1024,1025]
SMALL_EDGE = [1,2,3,4,7,8,9,15,16,17,31,32,33,63,64,65,95,96,97,127,128,129,255,256,257]
LAYOUTS = ["contig", "tcontig", "sliced"]
OPS = ["matmul_canary", "conv1x1_canary", "cumsum_inverse", "layout_cumsum", "sign_matmul", "pow2_matmul", "softmax_shift_gray"]

@dataclass(frozen=True)
class Plan:
    op: str
    n: int = 64
    k: int = 64
    m: int = 64
    c: int = 16
    h: int = 32
    w: int = 32
    layout: str = "contig"
    compiler: int = 0
    tf32: int = 0
    seed: int = 0
    source: str = "random"

    def cell(self):
        return f"{self.op}|n={bucket(self.n)}|k={bucket(self.k)}|m={bucket(self.m)}|c={bucket(self.c)}|h={bucket(self.h)}|layout={self.layout}|compile={self.compiler}|tf32={self.tf32}"

def bucket(x):
    x = int(x)
    if x <= 4: return "tiny"
    if x <= 17: return "edge16"
    if x <= 65: return "edge64"
    if x <= 129: return "edge128"
    if x <= 257: return "edge256"
    if x <= 513: return "edge512"
    if x <= 1025: return "edge1024"
    return "large"

def clamp_int(x, lo, hi):
    try:
        x = int(x)
    except Exception:
        x = lo
    return max(lo, min(hi, x))

def parse_plan_line(line: str):
    if not line or "PLAN" not in line.upper():
        return None
    kv = dict(re.findall(r"([a-zA-Z_][a-zA-Z0-9_]*)\s*=\s*([a-zA-Z0-9_+\-.]+)", line))
    op = kv.get("op", kv.get("OP", "")).strip()
    if op not in OPS:
        return None
    layout = kv.get("layout", "contig")
    if layout not in LAYOUTS:
        layout = "contig"
    n = clamp_int(kv.get("n", 64), 1, 1025)
    k = clamp_int(kv.get("k", 64), 1, 1025)
    m = clamp_int(kv.get("m", 64), 1, 1025)
    c = clamp_int(kv.get("c", 16), 1, 128)
    h = clamp_int(kv.get("h", 32), 1, 256)
    w = clamp_int(kv.get("w", h), 1, 256)
    compiler = 1 if str(kv.get("compiler", "0")).lower() in ("1","true","yes") else 0
    tf32 = 1 if str(kv.get("tf32", "0")).lower() in ("1","true","yes") else 0

    # hard cost guard
    if op in ("matmul_canary", "sign_matmul", "pow2_matmul"):
        if n * k * m > AXIOM_MAX_MATMUL_WORK:
            return None
    if op == "conv1x1_canary":
        if n * c * h * w > AXIOM_MAX_TENSOR_ELEMS:
            return None
    return Plan(op=op, n=n, k=k, m=m, c=c, h=h, w=w, layout=layout, compiler=compiler, tf32=tf32, seed=random.randrange(2**31), source="llm")

def random_plan():
    op = random.choices(
        OPS,
        weights=[34, 18, 17, 12, 8, 7, 4],
        k=1
    )[0]
    layout = random.choice(LAYOUTS)
    compiler = 1 if (CAN_COMPILE and random.random() < AXIOM_COMPILE_PROB) else 0
    tf32 = 1 if (torch.cuda.is_available() and random.random() < 0.025) else 0

    if op in ("matmul_canary", "sign_matmul", "pow2_matmul"):
        for _ in range(100):
            n = random.choice(EDGE)
            k = random.choice(EDGE)
            m = random.choice(EDGE)
            if n * k * m <= AXIOM_MAX_MATMUL_WORK:
                break
        return Plan(op=op, n=n, k=k, m=m, layout=layout, compiler=compiler, tf32=tf32, seed=random.randrange(2**31), source="random")

    if op == "conv1x1_canary":
        n = random.choice([1,2,3,4,7,8,9,16])
        c = random.choice([1,2,3,4,7,8,16,31,32,33,63,64])
        h = random.choice(SMALL_EDGE)
        w = random.choice(SMALL_EDGE)
        return Plan(op=op, n=n, c=c, h=h, w=w, layout=random.choice(["contig", "tcontig"]), compiler=compiler, tf32=tf32, seed=random.randrange(2**31), source="random")

    if op in ("cumsum_inverse", "layout_cumsum"):
        n = random.choice([1,2,3,4,7,8,16,31,32,33,64,65,127,128])
        k = random.choice(EDGE)
        return Plan(op=op, n=n, k=k, layout=layout, compiler=compiler, tf32=0, seed=random.randrange(2**31), source="random")

    # softmax gray
    n = random.choice([1,2,4,8,16,32,64,128])
    k = random.choice(SMALL_EDGE)
    return Plan(op=op, n=n, k=k, layout=layout, compiler=0, tf32=tf32, seed=random.randrange(2**31), source="random")

# -----------------------------
# Existing in-VRAM model as policy compiler
# -----------------------------
def find_existing_policy_model():
    g = globals()
    model_names = ["model", "policy_model", "gen_model", "llm_model", "MODEL"]
    tok_names = ["tokenizer", "tok", "processor"]
    mdl = None
    tok = None
    for name in model_names:
        obj = g.get(name, None)
        if obj is not None and hasattr(obj, "generate"):
            mdl = obj
            print(f"[policy] found model in globals(): {name}")
            break
    for name in tok_names:
        obj = g.get(name, None)
        if obj is None:
            continue
        if hasattr(obj, "apply_chat_template") or hasattr(obj, "__call__"):
            tok = obj
            print(f"[policy] found tokenizer/processor in globals(): {name}")
            break
        if hasattr(obj, "tokenizer"):
            tok = obj.tokenizer
            print(f"[policy] found processor.tokenizer in globals(): {name}.tokenizer")
            break
    return mdl, tok

POLICY_MODEL, POLICY_TOK = (None, None)
if AXIOM_USE_LLM_POLICY:
    POLICY_MODEL, POLICY_TOK = find_existing_policy_model()
else:
    print("[policy] disabled by AXIOM_USE_LLM_POLICY=0")

def policy_device():
    if POLICY_MODEL is None:
        return DEVICE
    try:
        return next(POLICY_MODEL.parameters()).device
    except Exception:
        return DEVICE

def llm_policy_plans(num_lines=64):
    if POLICY_MODEL is None or POLICY_TOK is None:
        return []
    prompt = f"""
You are a fuzzing policy compiler. Output only DSL lines, no prose.
Format:
PLAN op=<one of {OPS}> n=<edge size> k=<edge size> m=<edge size> c=<channels> h=<height> w=<width> layout=<contig|tcontig|sliced> compiler=<0|1> tf32=<0|1>

Task: propose high-value PyTorch strict-oracle tests after a 30-minute run found 0 strict bugs but many gray numeric drifts.
Prioritize exact theorem oracles: canary lanes, layout exactness, cumsum inverse, sign symmetry, power-of-two homogeneity, compiler/eager split.
Avoid invalid shapes. Use edge sizes around 31/32/33, 63/64/65, 127/128/129, 255/256/257, 511/512/513, 1023/1024/1025.
Output {num_lines} lines.
""".strip()

    try:
        if hasattr(POLICY_TOK, "apply_chat_template"):
            msgs = [{"role": "user", "content": prompt}]
            text = POLICY_TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            text = prompt

        enc = POLICY_TOK(text, return_tensors="pt")
        dev = policy_device()
        enc = {k: v.to(dev) for k, v in enc.items() if hasattr(v, "to")}

        with torch.no_grad():
            out = POLICY_MODEL.generate(
                **enc,
                max_new_tokens=900,
                do_sample=True,
                temperature=0.85,
                top_p=0.92,
                num_return_sequences=1,
                pad_token_id=getattr(POLICY_TOK, "eos_token_id", None),
            )
        decoded = POLICY_TOK.decode(out[0], skip_special_tokens=True)
        lines = decoded.splitlines()
        plans = []
        seen = set()
        for line in lines:
            p = parse_plan_line(line)
            if p is not None and p not in seen:
                plans.append(p)
                seen.add(p)
        print(f"[policy] parsed {len(plans)} usable plans from in-VRAM model")
        return plans[:num_lines]
    except Exception as e:
        print("[policy] generation failed; fallback to random DSL:", _short_exc(e))
        return []

policy_queue = deque(llm_policy_plans(AXIOM_POLICY_BOOTSTRAP_LINES))

# -----------------------------
# Tensor constructors
# -----------------------------
def make_matrix(rows, cols, layout="contig", positive=False, integer=False, seed=0, device=DEVICE):
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed) % (2**31 - 1))
    if integer:
        x = torch.randint(-3, 4, (rows, cols), generator=gen, dtype=torch.int32).to(torch.float32)
    elif positive:
        x = torch.rand((rows, cols), generator=gen, dtype=torch.float32) * 0.75 + 0.125
    else:
        x = torch.randn((rows, cols), generator=gen, dtype=torch.float32) * 0.5

    if layout == "tcontig":
        x = x.t().contiguous().t()
    elif layout == "sliced":
        if integer:
            base = torch.randint(-3, 4, (rows, cols * 2), generator=gen, dtype=torch.int32).to(torch.float32)
        elif positive:
            base = torch.rand((rows, cols * 2), generator=gen, dtype=torch.float32) * 0.75 + 0.125
        else:
            base = torch.randn((rows, cols * 2), generator=gen, dtype=torch.float32) * 0.5
        x = base[:, ::2]
    else:
        x = x.contiguous()
    return x.to(device)

def make_4d(n, c, h, w, layout="contig", seed=0, device=DEVICE):
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed) % (2**31 - 1))
    x = torch.rand((n, c, h, w), generator=gen, dtype=torch.float32) * 0.75 + 0.125
    if layout == "tcontig":
        x = x.contiguous(memory_format=torch.channels_last)
    return x.to(device)

# -----------------------------
# Compile wrappers
# -----------------------------
_COMPILED = {}

def maybe_compile(name, fn):
    if not CAN_COMPILE:
        return fn
    if name not in _COMPILED:
        _COMPILED[name] = torch.compile(fn, fullgraph=True, dynamic=False)
    return _COMPILED[name]

def matmul_fn(a, b):
    return a @ b

def conv1x1_fn(x, w):
    return F.conv2d(x, w, bias=None, stride=1, padding=0, dilation=1, groups=1)

def cumsum_fn(x):
    return torch.cumsum(x, dim=-1)

def softmax_fn(x):
    return torch.softmax(x, dim=-1)

# -----------------------------
# Signature and repro
# -----------------------------
def sig_hash(kind, plan, detail_core):
    payload = {
        "kind": kind,
        "plan": asdict(plan),
        "detail_core": detail_core,
        "torch": torch.__version__,
        "device": DEVICE,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

def repro_for_plan(plan: Plan, kind=""):
    return f"""# Repro generated by AXIOM++ final harvest
import torch, torch.nn.functional as F
torch.manual_seed({plan.seed})
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = {bool(plan.tf32)}
    torch.backends.cudnn.allow_tf32 = {bool(plan.tf32)}
# plan={asdict(plan)}
# finding_class={kind}
# Re-run the notebook cell's runner on this plan:
#   p = Plan(**{asdict(plan)!r})
#   run_plan_once(p)
"""

def finding(kind, plan, detail, strict=True):
    core = {
        "kind": kind,
        "op": plan.op,
        "shape": [plan.n, plan.k, plan.m, plan.c, plan.h, plan.w],
        "layout": plan.layout,
        "compiler": plan.compiler,
        "tf32": plan.tf32,
        "where": detail.get("where"),
        "row": detail.get("row"),
        "u": detail.get("u"),
        "exception": detail.get("exception"),
    }
    sig = sig_hash(kind, plan, core)
    return {
        "time": time.time(),
        "class": kind,
        "signature": sig,
        "strict": bool(strict),
        "plan": asdict(plan),
        "detail": detail,
        "repro": repro_for_plan(plan, kind),
    }

# -----------------------------
# Strict runners
# -----------------------------
def run_matmul_canary(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    out = []
    A = make_matrix(plan.n, plan.k, plan.layout, positive=True, integer=False, seed=plan.seed)
    B = make_matrix(plan.k, plan.m, plan.layout, positive=True, integer=False, seed=plan.seed + 17)

    # install canary rows: exact one-hot rows inside production-ish random matrix
    rows = sorted(set([0, plan.n // 2, plan.n - 1]))
    us = [0, plan.k // 2, plan.k - 1]
    for r, u in zip(rows, us):
        A[r, :] = 0.0
        A[r, int(u)] = 1.0

    try:
        C = matmul_fn(A, B)
        _sync()
    except Exception as e:
        return [finding("CRASH_EAGER_MATMUL", plan, {"exception": _short_exc(e)}, strict=True)]

    # TF32 true is a documented precision substitution map, not strict bug by default
    strict_canary = not bool(plan.tf32)
    for r, u in zip(rows, us):
        got = C[r, :]
        exp = B[int(u), :]
        if not bits_equal(got, exp):
            diff = first_bit_diff(got, exp)
            cls = "STRICT_CANARY_MATMUL" if strict_canary else "GRAY_TF32_CANARY_MATMUL"
            out.append(finding(cls, plan, {"where": "canary_row", "row": int(r), "u": int(u), "diff": diff, "max_abs": max_abs_diff(got, exp)}, strict=strict_canary))

    if plan.compiler and CAN_COMPILE:
        try:
            cfn = maybe_compile("matmul_fn", matmul_fn)
            Cc = cfn(A, B)
            _sync()
            for r, u in zip(rows, us):
                got = Cc[r, :]
                exp = B[int(u), :]
                if not bits_equal(got, exp) and not bool(plan.tf32):
                    diff = first_bit_diff(got, exp)
                    out.append(finding("STRICT_COMPILED_CANARY_MATMUL", plan, {"where": "compiled_canary_row", "row": int(r), "u": int(u), "diff": diff, "max_abs": max_abs_diff(got, exp)}, strict=True))
            if not bool(plan.tf32):
                # Compare only canary rows; full matmul may legitimately differ by reduction order outside exact canary rows
                for r in rows:
                    if not bits_equal(Cc[r, :], C[r, :]):
                        diff = first_bit_diff(Cc[r, :], C[r, :])
                        out.append(finding("STRICT_EAGER_COMPILE_CANARY_SPLIT", plan, {"where": "eager_compile_canary_row", "row": int(r), "diff": diff, "max_abs": max_abs_diff(Cc[r, :], C[r, :])}, strict=True))
        except Exception as e:
            msg = _short_exc(e)
            if _is_internal_compile_error(msg):
                out.append(finding("CRASH_COMPILE_MATMUL_INTERNAL", plan, {"exception": msg}, strict=True))
            else:
                out.append(finding("INVALID_COMPILE_MATMUL", plan, {"exception": msg}, strict=False))

    return out

def run_conv1x1_canary(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    out = []
    x = make_4d(plan.n, plan.c, plan.h, plan.w, plan.layout, seed=plan.seed)
    w = torch.zeros((plan.c, plan.c, 1, 1), dtype=torch.float32, device=DEVICE)
    idx = torch.arange(plan.c, device=DEVICE)
    w[idx, idx, 0, 0] = 1.0

    try:
        y = conv1x1_fn(x, w)
        _sync()
    except Exception as e:
        return [finding("CRASH_EAGER_CONV1X1", plan, {"exception": _short_exc(e)}, strict=True)]

    strict_canary = not bool(plan.tf32)
    if not bits_equal(y, x):
        cls = "STRICT_CANARY_CONV1X1" if strict_canary else "GRAY_TF32_CANARY_CONV1X1"
        out.append(finding(cls, plan, {"where": "identity_1x1_conv", "diff": first_bit_diff(y, x), "max_abs": max_abs_diff(y, x)}, strict=strict_canary))

    if plan.compiler and CAN_COMPILE:
        try:
            cfn = maybe_compile("conv1x1_fn", conv1x1_fn)
            yc = cfn(x, w)
            _sync()
            if not bits_equal(yc, x) and not bool(plan.tf32):
                out.append(finding("STRICT_COMPILED_CANARY_CONV1X1", plan, {"where": "compiled_identity_1x1_conv", "diff": first_bit_diff(yc, x), "max_abs": max_abs_diff(yc, x)}, strict=True))
            if not bits_equal(yc, y) and not bool(plan.tf32):
                out.append(finding("STRICT_EAGER_COMPILE_CONV1X1_SPLIT", plan, {"where": "compiled_vs_eager_identity_1x1_conv", "diff": first_bit_diff(yc, y), "max_abs": max_abs_diff(yc, y)}, strict=True))
        except Exception as e:
            msg = _short_exc(e)
            if _is_internal_compile_error(msg):
                out.append(finding("CRASH_COMPILE_CONV1X1_INTERNAL", plan, {"exception": msg}, strict=True))
            else:
                out.append(finding("INVALID_COMPILE_CONV1X1", plan, {"exception": msg}, strict=False))
    return out

def run_cumsum_inverse(plan: Plan):
    _set_tf32(False)
    torch.manual_seed(plan.seed)
    out = []
    x = make_matrix(plan.n, plan.k, plan.layout, integer=True, seed=plan.seed)
    # Bound is tiny: exact integer island
    try:
        y = cumsum_fn(x)
        z0 = torch.zeros_like(y[:, :1])
        rec = torch.diff(y, dim=-1, prepend=z0)
        _sync()
    except Exception as e:
        return [finding("CRASH_EAGER_CUMSUM", plan, {"exception": _short_exc(e)}, strict=True)]

    if not bits_equal(rec, x):
        out.append(finding("STRICT_CUMSUM_DIFF_INVERSE", plan, {"where": "diff(cumsum(x))", "diff": first_bit_diff(rec, x), "max_abs": max_abs_diff(rec, x)}, strict=True))

    # sign symmetry: cumsum(-x) must equal -cumsum(x) bitwise in exact island
    try:
        yn = cumsum_fn(-x)
        if not bits_equal(yn, -y):
            out.append(finding("STRICT_SIGN_SYMMETRY_CUMSUM", plan, {"where": "cumsum(-x)==-cumsum(x)", "diff": first_bit_diff(yn, -y), "max_abs": max_abs_diff(yn, -y)}, strict=True))
    except Exception as e:
        out.append(finding("CRASH_EAGER_CUMSUM_NEG", plan, {"exception": _short_exc(e)}, strict=True))

    if plan.compiler and CAN_COMPILE:
        try:
            cfn = maybe_compile("cumsum_fn", cumsum_fn)
            yc = cfn(x)
            z0c = torch.zeros_like(yc[:, :1])
            recc = torch.diff(yc, dim=-1, prepend=z0c)
            _sync()
            if not bits_equal(recc, x):
                out.append(finding("STRICT_COMPILED_CUMSUM_DIFF_INVERSE", plan, {"where": "compiled_diff(cumsum(x))", "diff": first_bit_diff(recc, x), "max_abs": max_abs_diff(recc, x)}, strict=True))
            if not bits_equal(yc, y):
                out.append(finding("STRICT_EAGER_COMPILE_CUMSUM_SPLIT", plan, {"where": "compiled_vs_eager_cumsum_exact_island", "diff": first_bit_diff(yc, y), "max_abs": max_abs_diff(yc, y)}, strict=True))
        except Exception as e:
            msg = _short_exc(e)
            if _is_internal_compile_error(msg):
                out.append(finding("CRASH_COMPILE_CUMSUM_INTERNAL", plan, {"exception": msg}, strict=True))
            else:
                out.append(finding("INVALID_COMPILE_CUMSUM", plan, {"exception": msg}, strict=False))
    return out

def run_layout_cumsum(plan: Plan):
    _set_tf32(False)
    torch.manual_seed(plan.seed)
    out = []
    base = make_matrix(plan.n, plan.k, "contig", integer=True, seed=plan.seed)
    alt = base.t().contiguous().t()
    try:
        y0 = cumsum_fn(base)
        y1 = cumsum_fn(alt)
        _sync()
    except Exception as e:
        return [finding("CRASH_LAYOUT_CUMSUM", plan, {"exception": _short_exc(e)}, strict=True)]
    if not bits_equal(y0, y1):
        out.append(finding("STRICT_LAYOUT_CUMSUM_SPLIT", plan, {"where": "contig_vs_transpose_contiguous_transpose", "diff": first_bit_diff(y0, y1), "max_abs": max_abs_diff(y0, y1)}, strict=True))
    return out

def run_sign_matmul(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    out = []
    A = make_matrix(plan.n, plan.k, plan.layout, positive=False, integer=True, seed=plan.seed)
    B = make_matrix(plan.k, plan.m, plan.layout, positive=False, integer=True, seed=plan.seed + 17)
    if plan.n * plan.k * plan.m > 2.0e8:
        # sign symmetry is useful, but do not waste huge compute here
        return []
    try:
        y = matmul_fn(A, B)
        yn = matmul_fn(-A, B)
        _sync()
    except Exception as e:
        return [finding("CRASH_SIGN_MATMUL", plan, {"exception": _short_exc(e)}, strict=True)]
    if not bits_equal(yn, -y) and not bool(plan.tf32):
        out.append(finding("STRICT_SIGN_SYMMETRY_MATMUL", plan, {"where": "(-A)@B == -(A@B)", "diff": first_bit_diff(yn, -y), "max_abs": max_abs_diff(yn, -y)}, strict=True))
    return out

def run_pow2_matmul(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    out = []
    if plan.n * plan.k * plan.m > 2.0e8:
        return []
    A = make_matrix(plan.n, plan.k, plan.layout, positive=False, integer=True, seed=plan.seed)
    B = make_matrix(plan.k, plan.m, plan.layout, positive=False, integer=True, seed=plan.seed + 17)
    s = random.choice([-3, -2, -1, 1, 2, 3])
    try:
        y = matmul_fn(A, B)
        ys = matmul_fn(torch.ldexp(A, torch.tensor(s, device=DEVICE)), B)
        exp = torch.ldexp(y, torch.tensor(s, device=DEVICE))
        _sync()
    except Exception as e:
        return [finding("CRASH_POW2_MATMUL", plan, {"exception": _short_exc(e)}, strict=True)]
    if not bits_equal(ys, exp) and not bool(plan.tf32):
        out.append(finding("STRICT_POW2_HOMOGENEITY_MATMUL", plan, {"where": "ldexp(A,s)@B == ldexp(A@B,s)", "scale": s, "diff": first_bit_diff(ys, exp), "max_abs": max_abs_diff(ys, exp)}, strict=True))
    return out

def run_softmax_shift_gray(plan: Plan):
    _set_tf32(bool(plan.tf32))
    torch.manual_seed(plan.seed)
    x = make_matrix(plan.n, plan.k, plan.layout, positive=False, integer=False, seed=plan.seed)
    c = torch.tensor(37.0, dtype=torch.float32, device=DEVICE)
    out = []
    try:
        y0 = softmax_fn(x)
        y1 = softmax_fn(x + c)
        _sync()
        mad = max_abs_diff(y0, y1)
        if mad > 1e-4:
            out.append(finding("GRAY_SOFTMAX_SHIFT_INVARIANCE", plan, {"where": "softmax(x+c)≈softmax(x)", "max_abs": mad}, strict=False))
    except Exception as e:
        out.append(finding("CRASH_SOFTMAX_SHIFT", plan, {"exception": _short_exc(e)}, strict=True))
    return out

def run_plan_once(plan: Plan):
    if plan.op == "matmul_canary":
        return run_matmul_canary(plan)
    if plan.op == "conv1x1_canary":
        return run_conv1x1_canary(plan)
    if plan.op == "cumsum_inverse":
        return run_cumsum_inverse(plan)
    if plan.op == "layout_cumsum":
        return run_layout_cumsum(plan)
    if plan.op == "sign_matmul":
        return run_sign_matmul(plan)
    if plan.op == "pow2_matmul":
        return run_pow2_matmul(plan)
    if plan.op == "softmax_shift_gray":
        return run_softmax_shift_gray(plan)
    return []

def stable_recheck(f0):
    """Re-run same plan. Keep strict only if the same signature/class is reproducible."""
    if not f0.get("strict", False):
        return False, []
    plan = Plan(**f0["plan"])
    same = 0
    seen = []
    for _ in range(max(1, AXIOM_RECHECK_N - 1)):
        try:
            fs = run_plan_once(plan)
        except Exception as e:
            fs = [finding("HARNESS_EXCEPTION", plan, {"exception": _short_exc(e), "trace": traceback.format_exc()[:600]}, strict=False)]
        seen.extend(fs)
        if any(x.get("signature") == f0.get("signature") and x.get("class") == f0.get("class") for x in fs):
            same += 1
    return same >= max(1, AXIOM_RECHECK_N - 1), seen

# -----------------------------
# Coverage-guided frontier loop
# -----------------------------
start = time.time()
tests = 0
coverage_cells = set()
status_mix = Counter()
class_mix = Counter()
strict_findings = {}
gray_findings = {}
invalid_examples = {}
novelty_score = defaultdict(float)

def choose_plan():
    # Use policy output mainly when it maps to a less-covered cell.
    if policy_queue and random.random() < 0.42:
        for _ in range(min(12, len(policy_queue))):
            p = policy_queue.popleft()
            if p.cell() not in coverage_cells or random.random() < 0.25:
                return p
            policy_queue.append(p)
    # novelty-biased random candidate sampling
    best = None
    best_s = -1e9
    for _ in range(8):
        p = random_plan()
        cell = p.cell()
        s = (3.0 if cell not in coverage_cells else 0.0) + random.random()
        # Push more canary/layout after 0-sig run; those are low false-positive.
        if p.op in ("matmul_canary", "conv1x1_canary", "cumsum_inverse", "layout_cumsum"):
            s += 0.5
        if p.compiler:
            s += 0.25
        if s > best_s:
            best, best_s = p, s
    return best

with open(OUT_JSONL, "w", encoding="utf-8") as jf:
    while time.time() - start < AXIOM_FINAL_BUDGET_SEC:
        p = choose_plan()
        tests += 1
        coverage_cells.add(p.cell())

        try:
            fs = run_plan_once(p)
            if not fs:
                status_mix["ok"] += 1
                continue

            emitted_any = False
            for f in fs:
                cls = f.get("class", "UNKNOWN")
                class_mix[cls] += 1

                if cls.startswith("INVALID"):
                    status_mix["invalid"] += 1
                    invalid_examples.setdefault(cls, f)
                    continue

                if cls.startswith("GRAY"):
                    status_mix["gray"] += 1
                    gray_findings.setdefault(f["signature"], f)
                    if AXIOM_GRAY_OPT_IN:
                        jf.write(json.dumps(f, ensure_ascii=False) + "\n")
                    continue

                if f.get("strict", False):
                    ok_stable, extra = stable_recheck(f)
                    if ok_stable:
                        status_mix["strict_signature"] += 1
                        strict_findings.setdefault(f["signature"], f)
                        jf.write(json.dumps(f, ensure_ascii=False) + "\n")
                        emitted_any = True
                    else:
                        status_mix["unstable_suppressed"] += 1
                        gray = dict(f)
                        gray["class"] = "UNSTABLE_SUPPRESSED_" + f.get("class", "UNKNOWN")
                        gray["strict"] = False
                        gray["recheck_seen_classes"] = Counter(x.get("class","?") for x in extra)
                        gray_findings.setdefault(gray["signature"], gray)
                else:
                    status_mix["non_strict"] += 1

            if not emitted_any and all(x.get("class","").startswith("GRAY") for x in fs):
                pass

        except RuntimeError as e:
            msg = _short_exc(e)
            status_mix["harness_runtime_error"] += 1
            if "out of memory" in msg.lower():
                status_mix["oom_suppressed"] += 1
                # Do NOT empty cache aggressively; model is in VRAM. Just collect Python refs.
                gc.collect()
            else:
                invalid_examples.setdefault("HARNESS_RUNTIME_ERROR", {"class": "HARNESS_RUNTIME_ERROR", "detail": msg, "plan": asdict(p)})
        except Exception as e:
            status_mix["harness_exception"] += 1
            invalid_examples.setdefault("HARNESS_EXCEPTION", {"class": "HARNESS_EXCEPTION", "detail": _short_exc(e), "trace": traceback.format_exc()[:900], "plan": asdict(p)})

        if tests % AXIOM_VERBOSE_EVERY == 0:
            elapsed = time.time() - start
            rate = tests / max(1e-9, elapsed)
            print(f"[{elapsed:7.1f}s] tests={tests} rate={rate:.1f}/s strict={len(strict_findings)} gray={len(gray_findings)} cells={len(coverage_cells)} mix={dict(status_mix)}")

# -----------------------------
# Final summary
# -----------------------------
elapsed = time.time() - start
rate = tests / max(1e-9, elapsed)

summary = {
    "run_id": RUN_ID,
    "elapsed_sec": elapsed,
    "tests": tests,
    "tests_per_sec": rate,
    "unique_strict_signatures": len(strict_findings),
    "unique_gray_signatures": len(gray_findings),
    "coverage_cells": len(coverage_cells),
    "status_mix": dict(status_mix),
    "class_mix": dict(class_mix),
    "strict_findings": list(strict_findings.values()),
    "gray_examples": list(gray_findings.values())[:20],
    "invalid_examples": invalid_examples,
    "jsonl": OUT_JSONL,
    "summary_md": OUT_SUMMARY,
    "torch": torch.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "compile": CAN_COMPILE,
    "notes": [
        "Strict signatures require theorem/bitwise oracle plus stability recheck.",
        "Gray drifts are suppressed by default; set AXIOM_GRAY_OPT_IN=1 only for heuristic triage.",
        "TF32 canary mismatch under tf32=1 is treated as gray precision-substitution map, not strict bug.",
        "No model is unloaded; this cell reuses an existing in-VRAM model only as a DSL policy compiler."
    ],
}

md_lines = []
md_lines.append(f"# AXIOM++ Final Harvest Summary — {RUN_ID}")
md_lines.append("")
md_lines.append(f"- elapsed_sec: `{elapsed:.2f}`")
md_lines.append(f"- tests: `{tests}`")
md_lines.append(f"- tests_per_sec: `{rate:.2f}`")
md_lines.append(f"- unique_strict_signatures: `{len(strict_findings)}`")
md_lines.append(f"- unique_gray_signatures: `{len(gray_findings)}`")
md_lines.append(f"- coverage_cells: `{len(coverage_cells)}`")
md_lines.append(f"- device: `{DEVICE}`")
md_lines.append(f"- gpu: `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}`")
md_lines.append("")
md_lines.append("## Status mix")
md_lines.append("```json")
md_lines.append(json.dumps(dict(status_mix), indent=2, ensure_ascii=False))
md_lines.append("```")
md_lines.append("")
md_lines.append("## Class mix")
md_lines.append("```json")
md_lines.append(json.dumps(dict(class_mix), indent=2, ensure_ascii=False))
md_lines.append("```")
md_lines.append("")
md_lines.append("## Strict findings")
if strict_findings:
    for sig, f in strict_findings.items():
        md_lines.append(f"### {sig} — {f['class']}")
        md_lines.append("```json")
        md_lines.append(json.dumps({"plan": f["plan"], "detail": f["detail"]}, indent=2, ensure_ascii=False, default=str))
        md_lines.append("```")
        md_lines.append("```python")
        md_lines.append(f["repro"])
        md_lines.append("```")
else:
    md_lines.append("No stable strict theorem/compile signatures found.")
    md_lines.append("")
    md_lines.append("Interpretation: this is not a failure. It means strict oracles did not produce evidence strong enough to report. Increase budget or enable gray triage separately; do not lower strictness.")

with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

AXIOM_FINAL_SUMMARY = summary
AXIOM_FINAL_FINDINGS = list(strict_findings.values())
AXIOM_FINAL_GRAY = list(gray_findings.values())

print("\n" + "=" * 88)
print(f"DONE AXIOM++ FINAL HARVEST: {tests} tests in {elapsed:.1f}s = {rate:.1f} test/s")
print(f"unique strict signatures: {len(strict_findings)}")
print(f"unique gray signatures suppressed: {len(gray_findings)}")
print(f"coverage cells: {len(coverage_cells)}")
print(f"status mix: {dict(status_mix)}")
print(f"class mix: {dict(class_mix)}")
print(f"jsonl: {OUT_JSONL}")
print(f"summary: {OUT_SUMMARY}")
print("=" * 88)

if strict_findings:
    print("\nTOP STRICT FINDINGS:")
    for i, f in enumerate(list(strict_findings.values())[:10], 1):
        print(f"\n[{i}] {f['signature']} {f['class']}")
        print("plan:", f["plan"])
        print("detail:", f["detail"])
else:
    print("\nNo stable strict signatures. Giữ bình tĩnh: ít nhất nó không biến nhiễu thành bug như một cái máy in báo nhầm.")
    print("Next move if still 0 after this cell:")
    print("  1) rerun with AXIOM_FINAL_BUDGET_SEC=3600")
    print("  2) set AXIOM_COMPILE_PROB=0.20 if torch.compile is the target")
    print("  3) only then set AXIOM_GRAY_OPT_IN=1 to export gray drift triage, NOT as bug report.")

In [ ]:
# === Report + save artifacts ===================================================
%matplotlib inline
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

if "RESULT" not in globals() or RESULT is None:
    print(">> RESULT is not defined; run the Assemble + run cell first.")
else:
    prefix = getattr(CFG, "checkpoint_prefix", "calm_12b_axiomplus")
    print("CALM-Fuzz | executor:", getattr(executor, "name", "?"), "| policy:", getattr(policy_llm, "name", "?"))
    print("unique fault signatures:", RESULT["seen"], "| tests:", RESULT["total"],
          "| %.0f test/s" % RESULT["test_per_s"], "| coverage cells:", RESULT["cells"],
          "| stop:", RESULT.get("stop_reason", ""))
    print("status mix:", RESULT["stats"])
    if RESULT.get("precision") is not None:
        print("ORACLE (vs injected ground truth): precision=%.3f recall=%.3f | baseline precision=%.3f | false alarms removed=%d"
              % (RESULT["precision"], RESULT["recall"], RESULT["baseline_precision"], RESULT["baseline_fp"]))

    df = pd.DataFrame(RESULT["bugs"])
    if len(df):
        cols = [c for c in ["op", "status", "detail", "steps_full", "steps_min", "src", "gt", "match"] if c in df.columns]
        print("\nunique reproducers (minimized):")
        print(df[cols].to_string(index=False))

    cc = np.array(RESULT["curve"], dtype=float)   # (total, unique_bugs, cells, tp, fp)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    if len(cc):
        ax[0].plot(cc[:, 0], cc[:, 1], color="crimson"); ax[0].set_title("unique fault signatures")
        ax[0].set_xlabel("tests"); ax[0].set_ylabel("count")
        ax[1].plot(cc[:, 0], cc[:, 2], color="teal"); ax[1].set_title("MAP-Elites coverage cells")
        ax[1].set_xlabel("tests"); ax[1].set_ylabel("cells")
        if cc.shape[1] >= 5 and cc[:, 3].max() + cc[:, 4].max() > 0:
            prec = cc[:, 3] / np.maximum(1, cc[:, 3] + cc[:, 4])
            ax[2].plot(cc[:, 0], prec, color="purple"); ax[2].set_ylim(0, 1.02)
            ax[2].set_title("CALM precision vs ground truth"); ax[2].set_xlabel("tests"); ax[2].set_ylabel("precision")
        else:
            ax[2].axis("off"); ax[2].set_title("(precision needs ground truth / MOCK)")
    plt.tight_layout(); plt.savefig(prefix + "_report.png", dpi=110)
    if "calm_copy_to_drive" in globals(): calm_copy_to_drive(prefix + "_report.png")
    plt.show()

    with open(prefix + "_bugs.jsonl", "w", encoding="utf-8") as f:
        for b in RESULT["bugs"]: f.write(json.dumps(b, ensure_ascii=False, default=str) + "\n")
    if "calm_copy_to_drive" in globals(): calm_copy_to_drive(prefix + "_bugs.jsonl")
    print("\n>> saved:", prefix + "_bugs.jsonl,", prefix + "_report.png")
    if globals().get("CALM_DRIVE_DIR"): print(">> mirrored to Drive:", CALM_DRIVE_DIR)


In [ ]:
# === Duplicate run guard =======================================================
# Older notebook revisions had a second full Assemble + run cell here, so
# Runtime > Run all would burn the T4 budget twice. This cell now only recovers
# if RESULT is genuinely missing.
if "RESULT" in globals() and RESULT is not None:
    print(">> RESULT already exists; skipping duplicate run cell.")
else:
    print(">> RESULT missing; running the guarded Assemble + run logic above is required before reports/snapshots.")


In [ ]:
# === Save full run snapshot ===================================================
import json, time, re
from pathlib import Path
from pprint import pformat

prefix = getattr(CFG, "checkpoint_prefix", "calm_12b_axiomplus")

def _safe_text(x):
    try:
        return pformat(x, width=160, compact=False)
    except Exception as e:
        return f"<<repr failed: {e}>>\n{repr(x)}"

def _safe_json_dump(obj):
    return json.dumps(obj, ensure_ascii=False, indent=2, default=str)


def _redact_obj(obj):
    """Recursively redact secrets before writing snapshots."""
    secret_keys = {"hf_token", "token", "access_token", "api_key", "password", "authorization"}
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            kl = str(k).lower()
            if any(s in kl for s in secret_keys):
                out[k] = "<<REDACTED>>" if v else ""
            else:
                out[k] = _redact_obj(v)
        return out
    if isinstance(obj, (list, tuple)):
        return [_redact_obj(x) for x in obj]
    if isinstance(obj, str):
        # Redact common Hugging Face token shape without changing normal text.
        return re.sub(r"hf_[A-Za-z0-9_\-]{12,}", "hf_<<REDACTED>>", obj)
    return obj

def _read_if_exists(path):
    p = Path(path)
    if p.exists():
        return p.read_text(encoding="utf-8", errors="replace")
    return ""

def _out_path(name):
    if "calm_local" in globals():
        return Path(calm_local(name))
    return Path(name)

saved_at = time.strftime("%Y-%m-%d %H:%M:%S")

progress_path = _out_path(prefix + "_progress.json")
bugs_path = _out_path(prefix + "_bugs_live.jsonl")

snapshot = {
    "saved_at": saved_at,
    "mode": "MOCK" if globals().get("MOCK") else "REAL",
    "executor": getattr(globals().get("executor"), "name", str(globals().get("executor", ""))),
    "policy_llm": getattr(globals().get("policy_llm"), "name", str(globals().get("policy_llm", ""))),
    "cfg": _redact_obj(vars(CFG) if hasattr(CFG, "__dict__") else str(CFG)),
    "result": _redact_obj(globals().get("RESULT", None)),
}

full_text = []
full_text.append(f"=== SAVED AT: {saved_at} ===\n")
full_text.append("=== RESULT ===\n")
full_text.append(_safe_text(_redact_obj(globals().get("RESULT", "<<RESULT not found>>"))))
full_text.append("\n\n=== PROGRESS JSON ===\n")
full_text.append(_redact_obj(_read_if_exists(progress_path) or "<<progress file not found>>"))
full_text.append("\n\n=== BUGS LIVE JSONL ===\n")
full_text.append(_redact_obj(_read_if_exists(bugs_path) or "<<bugs file not found>>"))

txt_out = _out_path(prefix + "_FULL_OUTPUT_SNAPSHOT.txt")
json_out = _out_path(prefix + "_FULL_OUTPUT_SNAPSHOT.json")

txt_out.parent.mkdir(parents=True, exist_ok=True)
txt_out.write_text("".join(full_text), encoding="utf-8")
json_out.write_text(_safe_json_dump(snapshot), encoding="utf-8")

if globals().get("CALM_DRIVE_DIR"):
    drive_dir = Path(CALM_DRIVE_DIR)
    drive_dir.mkdir(parents=True, exist_ok=True)
    (drive_dir / txt_out.name).write_text(txt_out.read_text(encoding="utf-8"), encoding="utf-8")
    (drive_dir / json_out.name).write_text(json_out.read_text(encoding="utf-8"), encoding="utf-8")

print("Saved full snapshot:")
print("TXT :", txt_out)
print("JSON:", json_out)
if globals().get("CALM_DRIVE_DIR"):
    print("Drive copies:", Path(CALM_DRIVE_DIR) / txt_out.name, "and", Path(CALM_DRIVE_DIR) / json_out.name)

In [ ]:
from pathlib import Path
import json

prefix = getattr(CFG, "checkpoint_prefix", "calm_12b_axiomplus")
roots = [Path("/content")]
if globals().get("CALM_DRIVE_DIR"):
    roots.append(Path(CALM_DRIVE_DIR))
files = []
for root in roots:
    files += [root / (prefix + "_FULL_OUTPUT_SNAPSHOT.txt"), root / (prefix + "_FULL_OUTPUT_SNAPSHOT.json"),
              root / (prefix + "_progress.json"), root / (prefix + "_bugs_live.jsonl")]

for p in files:
    print(f"\n{p}")
    print("exists:", p.exists())
    print("size:", p.stat().st_size if p.exists() else 0, "bytes")
    if p.exists() and p.suffix == ".json":
        json.loads(p.read_text(encoding="utf-8"))
        print("json: OK")


In [ ]:
# === Triage: re-run each reproducer to separate stable vs flaky faults =========
import json
from collections import Counter

rows = (globals().get("RESULT") or {}).get("bugs", [])
if not rows:
    print(">> No RESULT bugs to triage yet.")
else:
    def recheck(prog, repeats=5):
        got = []
        for _ in range(repeats):
            v = Oracle(CFG, executor).evaluate(prog)
            got.append((v.get("status"), str(v.get("detail", ""))[:80]))
        return got

    stable, flaky = [], []
    for r in rows:
        got = recheck(r["program"], repeats=5)
        c = Counter(s for s, _ in got)
        r2 = dict(r); r2["recheck"] = got; r2["recheck_counts"] = dict(c)
        (stable if c.get(r["status"], 0) >= 4 else flaky).append(r2)

    prefix = getattr(CFG, "checkpoint_prefix", "calm_12b_axiomplus")
    print("reproducers:", len(rows), "| stable:", len(stable), "| flaky:", len(flaky))
    print("stable by class:", Counter((r["op"], r["status"]) for r in stable))
    print("flaky  by class:", Counter((r["op"], r["status"]) for r in flaky))
    for name, data in [(prefix + "_stable.jsonl", stable), (prefix + "_flaky.jsonl", flaky)]:
        with open(name, "w", encoding="utf-8") as f:
            for r in data: f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
        if "calm_copy_to_drive" in globals(): calm_copy_to_drive(name)
    print(">> saved:", prefix + "_stable.jsonl,", prefix + "_flaky.jsonl")


In [ ]:

# === Optional AXIOM++ multi-seed fault-lab ablation ============================
# Set CALM_RUN_ABLATION=1 before opening the notebook to run this extra block.
import os, statistics, json, time
RUN_AXIOMPP_ABLATION = os.environ.get("CALM_RUN_ABLATION", "0") == "1"

if not RUN_AXIOMPP_ABLATION:
    print(">> Optional AXIOM++ ablation skipped. Set CALM_RUN_ABLATION=1 to run 5 short MOCK seeds.")
else:
    if not globals().get("MOCK"):
        print(">> Ablation is designed for FUZZ_MOCK=1; skipping in real framework mode.")
    else:
        rows = []
        old_budget = CFG.time_budget_s
        old_seed = CFG.rng_seed
        old_print = CFG.print_every
        try:
            CFG.time_budget_s = float(os.environ.get("CALM_ABLATION_BUDGET_S", "2.0"))
            CFG.print_every = 9999
            for seed in [7, 11, 17, 23, 31]:
                CFG.rng_seed = seed
                r = fuzz(CFG, executor=MockExecutor(), policy_llm=MockPolicyLLM(), quiet=True)
                rows.append({"seed": seed, "precision": r["precision"], "recall": r["recall"],
                             "baseline_precision": r["baseline_precision"], "baseline_fp": r["baseline_fp"],
                             "tests": r["total"], "uniq": r["seen"], "cells": r["cells"]})
            print(json.dumps(rows, ensure_ascii=False, indent=2))
            print("mean precision=%.3f recall=%.3f baseline_precision=%.3f"
                  % (statistics.mean(x["precision"] for x in rows),
                     statistics.mean(x["recall"] for x in rows),
                     statistics.mean(x["baseline_precision"] for x in rows)))
        finally:
            CFG.time_budget_s = old_budget
            CFG.rng_seed = old_seed
            CFG.print_every = old_print
